<a href="https://colab.research.google.com/github/Misha-private/Demo-repo/blob/main/Residual_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import os
def is_drive_mounted():
    return os.path.exists('/content/drive')
if not is_drive_mounted(): drive.mount('/content/drive')

Mounted at /content/drive


# Residual CNN

In [ ]:
# ============================================================
# train_rescnn_1layer_configurable_ai4dvar2.py
# ------------------------------------------------------------
# Configurable residual-CNN trainer for 1-layer SWE emulator
# on Klein beta-plane, with:
#   - multistep rollout data loss
#   - collocation state loss
#   - collocation tendency loss
#   - continuity residual (flux form)
#   - momentum residual (nonlinear)
#   - weak geostrophic-balance loss
#   - weak damping / smoothness loss
#
# Outputs go to:
#   /content/drive/MyDrive/AI_4DVAR2/...
# ============================================================

import os
import sys
import glob
import csv
import time
import random
import re
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# Import CollocBank
# ------------------------------------------------------------
sys.path.append("/content/drive/MyDrive/AI_4DVAR")
from colloc_bank import CollocBank

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# USER CONFIG
# ------------------------------------------------------------
class CFG:
    # -------- paths --------
    ROOT_IN  = "/content/drive/MyDrive/AI_4DVAR"
    ROOT_OUT = "/content/drive/MyDrive/AI_4DVAR2"

    DATA_DIR_CANDIDATES = [
        f"{ROOT_IN}/klein_ckpt_1L",
        f"{ROOT_IN}/klein_ckpt_AL_centers_colloc",
        f"{ROOT_IN}/klein_ckpt_AL_centers",
        f"{ROOT_IN}/klein_ckpt_1L_colloc",
    ]
    COLLOC_DIR = f"{ROOT_IN}/klein_ml_1L/colloc_raw"

    # customize this tag per experiment
    # EXP_NAME = "rescnn_b8_c96_lr1e4_t4_p4"
    # EXP_NAME = "rescnn_b8_c96_lr1e4_t4_p8"
    EXP_NAME = "rescnn_b8_c96_lr1e4_t6_p4"

    # -------- domain / physics --------
    NX = 256
    NY = 128
    Lx = 2.0e7
    Ly = 8.0e6
    H = 1000.0
    G = 9.81
    DT_MACRO = 200.0 * 30.0
    FMIN = 2.0e-5

    # -------- architecture --------
    N_BLOCKS = 8
    FEAT_CH  = 96
    HIDDEN   = 192

    # -------- optimization --------
    EPOCHS = 12
    BATCH_SIZE = 1
    LR = 1e-4
    WEIGHT_DECAY = 1e-6
    GRAD_CLIP = 1.0

    # -------- rollout --------
    ROLL_STEPS = 4
    ROLL_GAMMA = 0.95

    # -------- data loading --------
    MAX_WINDOWS_PER_IC = 4
    NUM_WORKERS = 0
    PIN_MEMORY = torch.cuda.is_available()

    # -------- collocation controls --------
    N_TIME_SLICES = 6
    PTS_PER_TIME  = 4

    # -------- loss weights --------
    LAMBDA_DATA       = 1.0
    LAMBDA_COLL_STATE = 0.05
    LAMBDA_COLL_TEND  = 0.1
    LAMBDA_CONT       = 0.2
    LAMBDA_MOM        = 0.5
    LAMBDA_GEO        = 0.01
    LAMBDA_SMOOTH     = 1e-4

    # -------- misc --------
    SAVE_EVERY_EPOCH = 1
    PRINT_BATCH_EVERY = 10
    RESUME_FROM = None

cfg = CFG()

cfg.DX = cfg.Lx / cfg.NX
cfg.DY = cfg.Ly / cfg.NY
cfg.CKPT_DIR = os.path.join(cfg.ROOT_OUT, "training_runs", cfg.EXP_NAME)
os.makedirs(cfg.CKPT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0
    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic = 0
    n_pairs = 0
    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += (len(files) - 1)
    return n_ic, n_pairs

def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir = None
    best_pairs = -1
    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_pairs = n_pairs
            best_dir = d
    if best_dir is None or best_pairs <= 0:
        raise RuntimeError("No valid snapshot root found.")
    print(f"[AutoDetect] using: {best_dir}")
    return best_dir

_step_re = re.compile(r"klein_step_(\d+)\.npz")

def extract_step_from_path(path):
    m = _step_re.search(os.path.basename(path))
    if m is None:
        raise ValueError(f"Could not parse step from {path}")
    return int(m.group(1))

def save_checkpoint(path, model, optimizer, epoch, loss_history, data_dir):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss_history": loss_history,
            "data_dir": data_dir,
            "exp_name": cfg.EXP_NAME,
            "config": {k: v for k, v in cfg.__dict__.items() if not k.startswith("__")},
        },
        path,
    )

def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    return ckpt.get("epoch", -1), ckpt.get("loss_history", []), ckpt

def save_loss_csv(path, loss_history):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow([
            "epoch", "train_total", "train_data",
            "train_coll_state", "train_coll_tend",
            "train_cont", "train_mom",
            "train_geo", "train_smooth"
        ])
        for row in loss_history:
            w.writerow(row)

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class SWWindowDataset(Dataset):
    def __init__(self, data_dir, roll_steps=4, max_windows_per_ic=None):
        self.samples = []

        ic_dirs = sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)])
        print(f"[Dataset] found {len(ic_dirs)} IC directories")

        for ic_dir in ic_dirs:
            ic_key = os.path.basename(ic_dir)
            files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
            print(f"[Dataset] {ic_key:20s} -> {len(files)} snapshot files")

            if len(files) < (roll_steps + 1):
                continue

            windows = []
            for i in range(len(files) - roll_steps):
                fseq = files[i:i + roll_steps + 1]
                macro_start_index = i + 1
                windows.append((fseq, ic_key, macro_start_index))

            if max_windows_per_ic is not None:
                windows = windows[:max_windows_per_ic]

            self.samples.extend(windows)

        if len(self.samples) == 0:
            raise RuntimeError("No usable training windows found.")

        print(f"[Dataset] total windows = {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fseq, ic_key, macro_start_index = self.samples[idx]

        seq = []
        times = []
        steps = []

        for f in fseq:
            z = np.load(f)
            seq.append(np.stack([z["eta"], z["uc"], z["vc"]], axis=0).astype(np.float32))
            times.append(float(z["t"]))
            steps.append(int(extract_step_from_path(f)))

        seq = np.stack(seq, axis=0)

        return {
            "seq": torch.from_numpy(seq),
            "times": np.array(times, dtype=np.float32),
            "steps": np.array(steps, dtype=np.int32),
            "ic_key": ic_key,
            "macro_start_index": np.int32(macro_start_index),
        }

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
class ResBlock(nn.Module):
    def __init__(self, ch, dilation=1):
        super().__init__()
        pad = dilation
        self.c1 = nn.Conv2d(ch, ch, 3, padding=pad, dilation=dilation)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=pad, dilation=dilation)
        self.act = nn.GELU()

    def forward(self, x):
        r = x
        x = self.act(self.c1(x))
        x = self.c2(x)
        return self.act(x + r)

class MultiStepContinuousResCNNModel(nn.Module):
    def __init__(self, in_ch=3, feat_ch=96, hidden=192, n_blocks=8):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
        )

        blocks = []
        dilations = [1, 1, 2, 2, 1, 1, 2, 2, 1, 1, 2, 2]
        for i in range(n_blocks):
            blocks.append(ResBlock(feat_ch, dilation=dilations[i % len(dilations)]))
        self.resnet = nn.Sequential(*blocks)

        self.grid_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, 3, 3, padding=1),
        )

        self.query_mlp = nn.Sequential(
            nn.Linear(feat_ch + 3 + 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

        # zero-init safe start
        nn.init.zeros_(self.grid_head[-1].weight)
        nn.init.zeros_(self.grid_head[-1].bias)
        nn.init.zeros_(self.query_mlp[-1].weight)
        nn.init.zeros_(self.query_mlp[-1].bias)

    def encode(self, x):
        return self.resnet(self.stem(x))

    def grid_tendency(self, feat):
        return self.grid_head(feat)

    def _sample_local(self, field_1b, x_norm_detached, y_norm_detached):
        grid = torch.stack([x_norm_detached, y_norm_detached], dim=-1).view(1, -1, 1, 2)
        vals = F.grid_sample(
            field_1b, grid,
            mode="bilinear",
            padding_mode="border",
            align_corners=True,
        )
        return vals.squeeze(0).squeeze(-1).transpose(0, 1)

    def query_state(self, x_1b, feat_1b, x_norm, y_norm, tau):
        x_norm_s = x_norm.detach()
        y_norm_s = y_norm.detach()

        feat_local = self._sample_local(feat_1b, x_norm_s, y_norm_s)
        state0_local = self._sample_local(x_1b, x_norm_s, y_norm_s)

        q = torch.stack([x_norm, y_norm, tau], dim=-1)
        inp = torch.cat([feat_local, state0_local, q], dim=-1)

        delta = self.query_mlp(inp)
        return state0_local + tau.unsqueeze(-1) * delta

# ------------------------------------------------------------
# Collocation sampling
# ------------------------------------------------------------
def sample_multitime_colloc(colloc_bank, ic_key, macro_index, n_time_slices=4, pts_per_time=4):
    n_request = max(n_time_slices * pts_per_time * 8, 64)
    base = colloc_bank.sample_nearest_macro(
        ic_key=ic_key,
        macro_index=macro_index,
        npts=n_request,
        replace=True,
    )

    t_sec = np.asarray(base["t_sec"]).reshape(-1)
    if t_sec.size == 0:
        return base

    order = np.argsort(t_sec)
    bins = np.array_split(order, n_time_slices)

    chosen = []
    for b in bins:
        if len(b) == 0:
            continue
        take = min(pts_per_time, len(b))
        idx = np.random.choice(b, size=take, replace=False)
        chosen.append(idx)

    if len(chosen) == 0:
        chosen = [order[:min(pts_per_time, len(order))]]

    chosen = np.concatenate(chosen, axis=0)

    out = {}
    for k, arr in base.items():
        arr = np.asarray(arr)
        if arr.ndim >= 1 and arr.shape[0] == len(t_sec):
            out[k] = arr[chosen]
        else:
            out[k] = arr
    return out

# ------------------------------------------------------------
# Derivative / balance helpers
# ------------------------------------------------------------
def safe_coriolis_torch(f, fmin):
    sign = torch.sign(f)
    sign = torch.where(sign == 0.0, torch.ones_like(sign), sign)
    return sign * torch.clamp(torch.abs(f), min=fmin)

# ------------------------------------------------------------
# Collocation losses
# ------------------------------------------------------------
def continuous_interval_losses_multitime(model, x_1b, feat_1b, colloc, t_start, dt_macro):
    x_m = torch.as_tensor(colloc["x_m"], dtype=torch.float32, device=device)
    y_m = torch.as_tensor(colloc["y_m"], dtype=torch.float32, device=device)
    t_sec = torch.as_tensor(colloc["t_sec"], dtype=torch.float32, device=device)

    tau = (t_sec - t_start) / dt_macro
    tau = torch.clamp(tau, 0.0, 1.0)

    x_norm = (2.0 * x_m / cfg.Lx) - 1.0
    y_norm = (2.0 * y_m / cfg.Ly) - 1.0

    x_norm = x_norm.clone().detach().requires_grad_(True)
    y_norm = y_norm.clone().detach().requires_grad_(True)
    tau    = tau.clone().detach().requires_grad_(True)

    state_hat = model.query_state(x_1b, feat_1b, x_norm, y_norm, tau)
    eta_hat = state_hat[:, 0]
    u_hat   = state_hat[:, 1]
    v_hat   = state_hat[:, 2]
    h_hat   = cfg.H + eta_hat

    def grads_of(field):
        gx, gy, gt = torch.autograd.grad(
            field.sum(), [x_norm, y_norm, tau],
            create_graph=True, retain_graph=True
        )
        return gx, gy, gt

    eta_xn, eta_yn, eta_tau = grads_of(eta_hat)
    u_xn,   u_yn,   u_tau   = grads_of(u_hat)
    v_xn,   v_yn,   v_tau   = grads_of(v_hat)

    eta_x = eta_xn * (2.0 / cfg.Lx)
    eta_y = eta_yn * (2.0 / cfg.Ly)
    h_x = eta_x
    h_y = eta_y

    u_x = u_xn * (2.0 / cfg.Lx)
    u_y = u_yn * (2.0 / cfg.Ly)
    v_x = v_xn * (2.0 / cfg.Lx)
    v_y = v_yn * (2.0 / cfg.Ly)

    eta_t = eta_tau / dt_macro
    h_t = eta_t
    u_t = u_tau / dt_macro
    v_t = v_tau / dt_macro

    hu = h_hat * u_hat
    hv = h_hat * v_hat
    hu_xn, _, _ = grads_of(hu)
    _, hv_yn, _ = grads_of(hv)
    hu_x = hu_xn * (2.0 / cfg.Lx)
    hv_y = hv_yn * (2.0 / cfg.Ly)

    eta_true = torch.as_tensor(colloc["eta"], dtype=torch.float32, device=device)
    u_true   = torch.as_tensor(colloc["uc"],  dtype=torch.float32, device=device)
    v_true   = torch.as_tensor(colloc["vc"],  dtype=torch.float32, device=device)

    deta_dt_fd = torch.as_tensor(colloc["deta_dt_fd"], dtype=torch.float32, device=device)
    duc_dt_fd  = torch.as_tensor(colloc["duc_dt_fd"],  dtype=torch.float32, device=device)
    dvc_dt_fd  = torch.as_tensor(colloc["dvc_dt_fd"],  dtype=torch.float32, device=device)

    f_fd = torch.as_tensor(colloc["f"], dtype=torch.float32, device=device)

    # state collocation
    loss_coll_state = ((eta_hat - eta_true) ** 2).mean()
    loss_coll_state += ((u_hat - u_true) ** 2).mean()
    loss_coll_state += ((v_hat - v_true) ** 2).mean()

    # tendency collocation
    loss_coll_tend = ((eta_t - deta_dt_fd) ** 2).mean()
    loss_coll_tend += ((u_t - duc_dt_fd) ** 2).mean()
    loss_coll_tend += ((v_t - dvc_dt_fd) ** 2).mean()

    # continuity in flux form
    resid_cont = h_t + hu_x + hv_y
    loss_cont = (resid_cont ** 2).mean()

    # nonlinear momentum
    adv_u = u_hat * u_x + v_hat * u_y
    adv_v = u_hat * v_x + v_hat * v_y
    resid_u = u_t + adv_u - f_fd * v_hat + cfg.G * h_x
    resid_v = v_t + adv_v + f_fd * u_hat + cfg.G * h_y
    loss_mom = (resid_u ** 2).mean() + (resid_v ** 2).mean()

    # weak geostrophic balance penalty
    f_safe = safe_coriolis_torch(f_fd, cfg.FMIN)
    u_g = -(cfg.G / f_safe) * eta_y
    v_g =  (cfg.G / f_safe) * eta_x
    loss_geo = ((u_hat - u_g) ** 2).mean() + ((v_hat - v_g) ** 2).mean()

    # weak smoothness / damping penalty
    loss_smooth = (u_x ** 2 + u_y ** 2 + v_x ** 2 + v_y ** 2).mean()

    return loss_coll_state, loss_coll_tend, loss_cont, loss_mom, loss_geo, loss_smooth

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
def train():
    data_dir = autodetect_data_dir(cfg.DATA_DIR_CANDIDATES)

    if not os.path.exists(cfg.COLLOC_DIR):
        raise RuntimeError(f"COLLOC_DIR does not exist: {cfg.COLLOC_DIR}")

    colloc_bank = CollocBank(cfg.COLLOC_DIR, verbose=True)

    dataset = SWWindowDataset(
        data_dir,
        roll_steps=cfg.ROLL_STEPS,
        max_windows_per_ic=cfg.MAX_WINDOWS_PER_IC,
    )

    loader = DataLoader(
        dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=cfg.PIN_MEMORY,
        drop_last=False,
    )

    model = MultiStepContinuousResCNNModel(
        feat_ch=cfg.FEAT_CH,
        hidden=cfg.HIDDEN,
        n_blocks=cfg.N_BLOCKS,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)

    start_epoch = 0
    loss_history = []

    if cfg.RESUME_FROM is not None and os.path.exists(cfg.RESUME_FROM):
        print(f"[Resume] loading checkpoint: {cfg.RESUME_FROM}")
        loaded_epoch, loaded_history, _ = load_checkpoint(cfg.RESUME_FROM, model, optimizer)
        start_epoch = loaded_epoch + 1
        loss_history = loaded_history

    print(f"[Train] output dir            = {cfg.CKPT_DIR}")
    print(f"[Train] windows               = {len(dataset)}")
    print(f"[Train] batches/epoch         = {len(loader)}")
    print(f"[Train] N_BLOCKS              = {cfg.N_BLOCKS}")
    print(f"[Train] FEAT_CH               = {cfg.FEAT_CH}")
    print(f"[Train] LR                    = {cfg.LR}")
    print(f"[Train] N_TIME_SLICES         = {cfg.N_TIME_SLICES}")
    print(f"[Train] PTS_PER_TIME          = {cfg.PTS_PER_TIME}")
    print(f"[Train] colloc/interval       = {cfg.N_TIME_SLICES * cfg.PTS_PER_TIME}")

    for epoch in range(start_epoch, cfg.EPOCHS):
        t0 = time.time()
        model.train()

        run_total = 0.0
        run_data = 0.0
        run_cstate = 0.0
        run_ctend = 0.0
        run_cont = 0.0
        run_mom = 0.0
        run_geo = 0.0
        run_smooth = 0.0

        for ib, batch in enumerate(loader):
            seq = batch["seq"].to(device, non_blocking=True)
            times = batch["times"].numpy()
            ic_keys = batch["ic_key"]
            macro_start = batch["macro_start_index"].numpy()

            B = seq.shape[0]
            optimizer.zero_grad(set_to_none=True)

            loss_data = 0.0
            loss_cstate = 0.0
            loss_ctend = 0.0
            loss_cont = 0.0
            loss_mom = 0.0
            loss_geo = 0.0
            loss_smooth = 0.0
            n_phys = 0

            x = seq[:, 0]

            for k in range(cfg.ROLL_STEPS):
                feat = model.encode(x)
                xdot_grid = model.grid_tendency(feat)
                x_next = x + cfg.DT_MACRO * xdot_grid

                truth_next = seq[:, k + 1]
                loss_data = loss_data + (cfg.ROLL_GAMMA ** k) * ((x_next - truth_next) ** 2).mean()

                for b in range(B):
                    ic_key = ic_keys[b]
                    macro_idx = int(macro_start[b] + k)
                    t_start = float(times[b, k])

                    if not colloc_bank.has_ic(ic_key):
                        continue

                    colloc = sample_multitime_colloc(
                        colloc_bank=colloc_bank,
                        ic_key=ic_key,
                        macro_index=macro_idx,
                        n_time_slices=cfg.N_TIME_SLICES,
                        pts_per_time=cfg.PTS_PER_TIME,
                    )

                    l_cstate, l_ctend, l_cont, l_mom, l_geo, l_smooth = continuous_interval_losses_multitime(
                        model=model,
                        x_1b=x[b:b+1],
                        feat_1b=feat[b:b+1],
                        colloc=colloc,
                        t_start=t_start,
                        dt_macro=cfg.DT_MACRO,
                    )

                    loss_cstate += l_cstate
                    loss_ctend += l_ctend
                    loss_cont += l_cont
                    loss_mom += l_mom
                    loss_geo += l_geo
                    loss_smooth += l_smooth
                    n_phys += 1

                x = x_next

            if n_phys > 0:
                loss_cstate = loss_cstate / n_phys
                loss_ctend = loss_ctend / n_phys
                loss_cont = loss_cont / n_phys
                loss_mom = loss_mom / n_phys
                loss_geo = loss_geo / n_phys
                loss_smooth = loss_smooth / n_phys
            else:
                zero = torch.tensor(0.0, device=device)
                loss_cstate = zero
                loss_ctend = zero
                loss_cont = zero
                loss_mom = zero
                loss_geo = zero
                loss_smooth = zero

            loss = (
                cfg.LAMBDA_DATA       * loss_data
                + cfg.LAMBDA_COLL_STATE * loss_cstate
                + cfg.LAMBDA_COLL_TEND  * loss_ctend
                + cfg.LAMBDA_CONT       * loss_cont
                + cfg.LAMBDA_MOM        * loss_mom
                + cfg.LAMBDA_GEO        * loss_geo
                + cfg.LAMBDA_SMOOTH     * loss_smooth
            )

            if not torch.isfinite(loss):
                print("Non-finite diagnostics:")
                print("  data   =", float(loss_data.detach().cpu()))
                print("  cstate =", float(loss_cstate.detach().cpu()))
                print("  ctend  =", float(loss_ctend.detach().cpu()))
                print("  cont   =", float(loss_cont.detach().cpu()))
                print("  mom    =", float(loss_mom.detach().cpu()))
                print("  geo    =", float(loss_geo.detach().cpu()))
                print("  smooth =", float(loss_smooth.detach().cpu()))
                raise RuntimeError(f"Non-finite loss at epoch={epoch}, batch={ib}: {loss.item()}")

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            optimizer.step()

            run_total += float(loss.detach().cpu())
            run_data += float(loss_data.detach().cpu())
            run_cstate += float(loss_cstate.detach().cpu())
            run_ctend += float(loss_ctend.detach().cpu())
            run_cont += float(loss_cont.detach().cpu())
            run_mom += float(loss_mom.detach().cpu())
            run_geo += float(loss_geo.detach().cpu())
            run_smooth += float(loss_smooth.detach().cpu())

            if (ib % cfg.PRINT_BATCH_EVERY) == 0:
                print(
                    f"[epoch {epoch:03d} | batch {ib:04d}/{len(loader):04d}] "
                    f"total={float(loss.detach().cpu()):.6e} "
                    f"data={float(loss_data.detach().cpu()):.6e} "
                    f"cstate={float(loss_cstate.detach().cpu()):.6e} "
                    f"ctend={float(loss_ctend.detach().cpu()):.6e} "
                    f"cont={float(loss_cont.detach().cpu()):.6e} "
                    f"mom={float(loss_mom.detach().cpu()):.6e} "
                    f"geo={float(loss_geo.detach().cpu()):.6e} "
                    f"smooth={float(loss_smooth.detach().cpu()):.6e}"
                )

        n_batches = max(len(loader), 1)
        ep_total = run_total / n_batches
        ep_data = run_data / n_batches
        ep_cstate = run_cstate / n_batches
        ep_ctend = run_ctend / n_batches
        ep_cont = run_cont / n_batches
        ep_mom = run_mom / n_batches
        ep_geo = run_geo / n_batches
        ep_smooth = run_smooth / n_batches

        loss_history.append([
            epoch, ep_total, ep_data, ep_cstate, ep_ctend, ep_cont, ep_mom, ep_geo, ep_smooth
        ])

        print(
            f"Epoch {epoch:03d} done | "
            f"total={ep_total:.6e} | data={ep_data:.6e} | "
            f"cstate={ep_cstate:.6e} | ctend={ep_ctend:.6e} | "
            f"cont={ep_cont:.6e} | mom={ep_mom:.6e} | "
            f"geo={ep_geo:.6e} | smooth={ep_smooth:.6e} | "
            f"time={time.time() - t0:.1f}s"
        )

        last_ckpt = os.path.join(cfg.CKPT_DIR, "last_model.pt")
        save_checkpoint(last_ckpt, model, optimizer, epoch, loss_history, data_dir)

        if ((epoch + 1) % cfg.SAVE_EVERY_EPOCH) == 0:
            save_checkpoint(
                os.path.join(cfg.CKPT_DIR, f"model_epoch_{epoch:03d}.pt"),
                model, optimizer, epoch, loss_history, data_dir
            )

        save_loss_csv(os.path.join(cfg.CKPT_DIR, "loss_history.csv"), loss_history)

    final_ckpt = os.path.join(cfg.CKPT_DIR, "final_model.pt")
    save_checkpoint(final_ckpt, model, optimizer, cfg.EPOCHS - 1, loss_history, data_dir)
    save_loss_csv(os.path.join(cfg.CKPT_DIR, "loss_history.csv"), loss_history)

    print("[Train] finished.")
    print(f"[Train] final checkpoint: {final_ckpt}")

if __name__ == "__main__":
    train()

Using device: cuda
GPU: Tesla T4
[AutoDetect] checking candidate snapshot roots...
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
     valid IC dirs = 28, total pairs = 168
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers_colloc
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L_colloc
     valid IC dirs = 0, total pairs = 0
[AutoDetect] using: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
[CollocBank] loaded gauss_00 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_01 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_02 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_03 | samples=15616 | macro_groups=7


KeyboardInterrupt: 

# Evaluation with nudging adjusted for last three runs

In [ ]:
# ============================================================
# eval_rescnn_1layer_compare_3runs.py
# ------------------------------------------------------------
# Compare three ResCNN 1-layer SWE runs:
#   - b8_c96_t4_p4
#   - b8_c96_t4_p8
#   - b8_c96_t6_p4
#
# Features:
#   - exact residual CNN architecture matching training
#   - compares multiple checkpoints in one script
#   - plain rollout
#   - rollout + u,v diffusion
#   - rollout + weak geostrophic nudging
#   - summary CSV
#   - RMSE vs time plots
#   - KE vs time plots
#   - optional field plots
#
# Output:
#   /content/drive/MyDrive/AI_4DVAR2/eval_rescnn_compare_3runs
# ============================================================

import os
import re
import glob
import csv
import math
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
ROOT_IN  = "/content/drive/MyDrive/AI_4DVAR"
ROOT_OUT = "/content/drive/MyDrive/AI_4DVAR2"

DATA_DIR_CANDIDATES = [
    f"{ROOT_IN}/klein_ckpt_1L",
    f"{ROOT_IN}/klein_ckpt_AL_centers_colloc",
    f"{ROOT_IN}/klein_ckpt_AL_centers",
    f"{ROOT_IN}/klein_ckpt_1L_colloc",
]

# ------------------------------------------------------------
# USER: define the three runs here
# ------------------------------------------------------------
RUNS = [
    {
        "name": "b8_c96_t4_p4",
        "ckpt_dir": f"{ROOT_OUT}/training_runs/rescnn_b8_c96_lr1e4_t4_p4",
        "ckpt_name": "final_model.pt",
    },
    {
        "name": "b8_c96_t4_p8",
        "ckpt_dir": f"{ROOT_OUT}/training_runs/rescnn_b8_c96_lr1e4_t4_p8",
        "ckpt_name": "final_model.pt",
    },
    {
        "name": "b8_c96_t6_p4",
        "ckpt_dir": f"{ROOT_OUT}/training_runs/rescnn_b8_c96_lr1e4_t6_p4",
        "ckpt_name": "final_model.pt",
    },
]

EVAL_DIR = os.path.join(ROOT_OUT, "eval_rescnn_compare_3runs")
os.makedirs(EVAL_DIR, exist_ok=True)

# ------------------------------------------------------------
# Constants
# ------------------------------------------------------------
H = 1000.0
G = 9.81
DT_MACRO = 200.0 * 30.0

Lx = 2.0e7
Ly = 8.0e6
NX = 256
NY = 128
DX = Lx / NX
DY = Ly / NY

# for geostrophic nudging
FMIN = 2.0e-5

# ------------------------------------------------------------
# USER CONTROLS
# ------------------------------------------------------------
# MODE: "one", "list", "all"
MODE = "list"

ONE_CASE = "gauss_00"

CASE_LIST = [
    "gauss_00",
    "test_modon_00",
    "test_rh_00",
    "wave_00",
]

START_INDEX = 0
ROLLOUT_STEPS = 6

MAKE_FIELD_PLOTS = False
FIELD_PLOT_RUN_NAME = "b8_c96_t6_p4"
FIELD_PLOT_METHOD = "plain"
FIELD_PLOT_STEPS = [0, 2, 5]  # rollout indices, 0-based over prediction steps

# Which inference methods to test
RUN_PLAIN = True
RUN_DIFFUSION = True
RUN_GEONUDGE = True

# diffusion strengths
NU_LIST = [5.0e4, 1.0e5, 2.0e5]

# geostrophic nudging strengths
ALPHA_LIST = [0.05, 0.10, 0.20]

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0
    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic = 0
    n_pairs = 0
    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += (len(files) - 1)
    return n_ic, n_pairs

def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir = None
    best_pairs = -1
    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_pairs = n_pairs
            best_dir = d
    if best_dir is None or best_pairs <= 0:
        raise RuntimeError("No valid snapshot root found.")
    print(f"[AutoDetect] using: {best_dir}")
    return best_dir

_step_re = re.compile(r"klein_step_(\d+)\.npz")

def extract_step_from_path(path):
    m = _step_re.search(os.path.basename(path))
    if m is None:
        raise ValueError(f"Could not parse step from {path}")
    return int(m.group(1))

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def domain_mean_ke(eta, u, v, H=1000.0):
    h = H + eta
    ke = 0.5 * h * (u*u + v*v)
    return float(np.mean(ke))

def total_rmse_from_components(rmse_eta, rmse_u, rmse_v):
    return np.sqrt(rmse_eta**2 + rmse_u**2 + rmse_v**2)

def list_ic_dirs(data_dir):
    return sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)])

def list_ic_keys(data_dir):
    return [os.path.basename(d) for d in list_ic_dirs(data_dir)]

def sanitize_float_for_name(x):
    s = f"{x:.3e}"
    s = s.replace("+", "")
    s = s.replace(".", "p")
    s = s.replace("-", "m")
    return s

# ------------------------------------------------------------
# Discrete operators
# ------------------------------------------------------------
def laplacian_np(field, dx, dy):
    left  = np.roll(field,  1, axis=1)
    right = np.roll(field, -1, axis=1)
    d2x = (left - 2.0 * field + right) / (dx * dx)

    d2y = np.zeros_like(field)
    d2y[1:-1, :] = (field[2:, :] - 2.0 * field[1:-1, :] + field[:-2, :]) / (dy * dy)
    d2y[0, :]    = (field[1, :] - field[0, :]) / (dy * dy)
    d2y[-1, :]   = (field[-2, :] - field[-1, :]) / (dy * dy)

    return d2x + d2y

def ddx_np(field, dx):
    left  = np.roll(field,  1, axis=1)
    right = np.roll(field, -1, axis=1)
    return (right - left) / (2.0 * dx)

def ddy_np(field, dy):
    out = np.zeros_like(field)
    out[1:-1, :] = (field[2:, :] - field[:-2, :]) / (2.0 * dy)
    out[0, :]    = (field[1, :] - field[0, :]) / dy
    out[-1, :]   = (field[-1, :] - field[-2, :]) / dy
    return out

def safe_coriolis(f, fmin=2.0e-5):
    sign = np.sign(f)
    sign[sign == 0.0] = 1.0
    return sign * np.maximum(np.abs(f), fmin)

def geostrophic_velocity_from_eta(eta, f, dx, dy, g=9.81, fmin=2.0e-5):
    etax = ddx_np(eta, dx)
    etay = ddy_np(eta, dy)
    f_safe = safe_coriolis(f, fmin=fmin)
    ug = -(g / f_safe) * etay
    vg =  (g / f_safe) * etax
    return ug, vg

# ------------------------------------------------------------
# Exact residual CNN architecture from training
# ------------------------------------------------------------
class ResBlock(nn.Module):
    def __init__(self, ch, dilation=1):
        super().__init__()
        pad = dilation
        self.c1 = nn.Conv2d(ch, ch, kernel_size=3, padding=pad, dilation=dilation)
        self.c2 = nn.Conv2d(ch, ch, kernel_size=3, padding=pad, dilation=dilation)
        self.act = nn.GELU()

    def forward(self, x):
        r = x
        x = self.act(self.c1(x))
        x = self.c2(x)
        return self.act(x + r)

class MultiStepContinuousResCNNModel(nn.Module):
    def __init__(self, in_ch=3, feat_ch=96, hidden=192, n_blocks=8):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, feat_ch, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, kernel_size=3, padding=1),
            nn.GELU(),
        )

        blocks = []
        dilations = [1, 1, 2, 2, 1, 1, 2, 2]
        for i in range(n_blocks):
            blocks.append(ResBlock(feat_ch, dilation=dilations[i % len(dilations)]))
        self.resnet = nn.Sequential(*blocks)

        self.grid_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, 3, kernel_size=3, padding=1),
        )

        self.query_mlp = nn.Sequential(
            nn.Linear(feat_ch + 3 + 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

    def encode(self, x):
        x = self.stem(x)
        x = self.resnet(x)
        return x

    def grid_tendency(self, feat):
        return self.grid_head(feat)

    def _sample_local(self, field_1b, x_norm_detached, y_norm_detached):
        grid = torch.stack([x_norm_detached, y_norm_detached], dim=-1).view(1, -1, 1, 2)
        vals = F.grid_sample(
            field_1b, grid,
            mode="bilinear",
            padding_mode="border",
            align_corners=True
        )
        vals = vals.squeeze(0).squeeze(-1).transpose(0, 1)
        return vals

    def query_state(self, x_1b, feat_1b, x_norm, y_norm, tau):
        x_norm_s = x_norm.detach()
        y_norm_s = y_norm.detach()

        feat_local = self._sample_local(feat_1b, x_norm_s, y_norm_s)
        state0_local = self._sample_local(x_1b, x_norm_s, y_norm_s)

        q = torch.stack([x_norm, y_norm, tau], dim=-1)
        inp = torch.cat([feat_local, state0_local, q], dim=-1)

        delta = self.query_mlp(inp)
        state_hat = state0_local + tau.unsqueeze(-1) * delta
        return state_hat

# ------------------------------------------------------------
# Load all checkpoints
# ------------------------------------------------------------
def load_all_models(run_defs):
    loaded = []
    for rd in run_defs:
        ckpt_path = os.path.join(rd["ckpt_dir"], rd["ckpt_name"])
        if not os.path.exists(ckpt_path):
            raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

        model = MultiStepContinuousResCNNModel().to(device)
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        model.eval()

        data_dir = ckpt.get("data_dir", None)

        loaded.append({
            "name": rd["name"],
            "ckpt_path": ckpt_path,
            "data_dir_from_ckpt": data_dir,
            "model": model,
        })

        print(f"[Load] {rd['name']} <- {ckpt_path}")
    return loaded

loaded_runs = load_all_models(RUNS)

# ------------------------------------------------------------
# Locate FD data
# ------------------------------------------------------------
data_dir = None
for rr in loaded_runs:
    dd = rr.get("data_dir_from_ckpt", None)
    if (dd is not None) and os.path.exists(dd):
        data_dir = dd
        break

if data_dir is None:
    data_dir = autodetect_data_dir(DATA_DIR_CANDIDATES)

print("[Eval] using FD data dir:", data_dir)

available_keys = list_ic_keys(data_dir)
print(f"[Eval] found {len(available_keys)} ICs")
print("[Eval] available ICs:", available_keys)

# ------------------------------------------------------------
# Select cases
# ------------------------------------------------------------
if MODE == "one":
    selected_cases = [ONE_CASE]
elif MODE == "list":
    selected_cases = CASE_LIST[:]
elif MODE == "all":
    selected_cases = available_keys[:]
else:
    raise ValueError(f"Unknown MODE={MODE}")

selected_cases = [k for k in selected_cases if k in available_keys]
if len(selected_cases) == 0:
    raise RuntimeError("No valid selected cases found.")

print("[Eval] selected cases:", selected_cases)

# ------------------------------------------------------------
# Truth loader
# ------------------------------------------------------------
def load_truth_case(ic_key):
    ic_dir = os.path.join(data_dir, ic_key)
    files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))

    n_avail = len(files) - START_INDEX - 1
    if n_avail <= 0:
        return None

    rollout_steps = min(ROLLOUT_STEPS, n_avail)

    truth_seq = []
    truth_times = []
    truth_steps = []
    f_ref = None

    for fpath in files[START_INDEX:START_INDEX + rollout_steps + 1]:
        z = np.load(fpath)
        truth_seq.append(np.stack([z["eta"], z["uc"], z["vc"]], axis=0).astype(np.float32))
        truth_times.append(float(z["t"]))
        truth_steps.append(int(extract_step_from_path(fpath)))
        if f_ref is None:
            f_ref = z["f"].astype(np.float32)

    truth_seq = np.stack(truth_seq, axis=0)
    truth_times = np.array(truth_times, dtype=np.float32)
    truth_steps = np.array(truth_steps, dtype=np.int32)

    roll_steps = np.arange(1, rollout_steps + 1)
    roll_hours = roll_steps * DT_MACRO / 3600.0

    return {
        "ic_key": ic_key,
        "rollout_steps": rollout_steps,
        "truth_seq": truth_seq,
        "truth_times": truth_times,
        "truth_steps": truth_steps,
        "roll_hours": roll_hours,
        "f_ref": f_ref,
    }

# ------------------------------------------------------------
# Rollout
# ------------------------------------------------------------
def rollout_case(model, truth_pack, method_name="plain", nu=0.0, alpha=0.0):
    truth_seq = truth_pack["truth_seq"]
    rollout_steps = truth_pack["rollout_steps"]
    f_ref = truth_pack["f_ref"]
    roll_hours = truth_pack["roll_hours"]

    x = torch.tensor(truth_seq[0], dtype=torch.float32, device=device).unsqueeze(0)
    pred_seq = [truth_seq[0].copy()]

    with torch.no_grad():
        for k in range(rollout_steps):
            feat = model.encode(x)
            xdot = model.grid_tendency(feat)

            x_np = x[0].detach().cpu().numpy()
            xdot_np = xdot[0].detach().cpu().numpy()

            eta = x_np[0]
            u   = x_np[1]
            v   = x_np[2]

            eta_dot = xdot_np[0]
            u_dot   = xdot_np[1]
            v_dot   = xdot_np[2]

            eta_ml = eta + DT_MACRO * eta_dot
            u_ml   = u   + DT_MACRO * u_dot
            v_ml   = v   + DT_MACRO * v_dot

            if method_name == "diffusion":
                u_ml = u_ml + DT_MACRO * nu * laplacian_np(u, DX, DY)
                v_ml = v_ml + DT_MACRO * nu * laplacian_np(v, DX, DY)

            elif method_name == "geonudge":
                ug, vg = geostrophic_velocity_from_eta(
                    eta_ml, f_ref, DX, DY, g=G, fmin=FMIN
                )
                u_ml = (1.0 - alpha) * u_ml + alpha * ug
                v_ml = (1.0 - alpha) * v_ml + alpha * vg

            x_next_np = np.empty_like(x_np)
            x_next_np[0] = eta_ml
            x_next_np[1] = u_ml
            x_next_np[2] = v_ml

            x = torch.tensor(x_next_np, dtype=torch.float32, device=device).unsqueeze(0)
            pred_seq.append(x_next_np.copy())

    pred_seq = np.stack(pred_seq, axis=0)

    rmse_eta = []
    rmse_u   = []
    rmse_v   = []
    ke_fd    = []
    ke_ml    = []

    for k in range(1, rollout_steps + 1):
        eta_fd = truth_seq[k, 0]
        u_fd   = truth_seq[k, 1]
        v_fd   = truth_seq[k, 2]

        eta_ml = pred_seq[k, 0]
        u_ml   = pred_seq[k, 1]
        v_ml   = pred_seq[k, 2]

        rmse_eta.append(rmse(eta_ml, eta_fd))
        rmse_u.append(rmse(u_ml, u_fd))
        rmse_v.append(rmse(v_ml, v_fd))

        ke_fd.append(domain_mean_ke(eta_fd, u_fd, v_fd, H=H))
        ke_ml.append(domain_mean_ke(eta_ml, u_ml, v_ml, H=H))

    rmse_eta = np.array(rmse_eta)
    rmse_u   = np.array(rmse_u)
    rmse_v   = np.array(rmse_v)
    ke_fd    = np.array(ke_fd)
    ke_ml    = np.array(ke_ml)

    return {
        "method": method_name,
        "nu": nu,
        "alpha": alpha,
        "roll_hours": roll_hours,
        "truth_seq": truth_seq,
        "pred_seq": pred_seq,
        "rmse_eta": rmse_eta,
        "rmse_u": rmse_u,
        "rmse_v": rmse_v,
        "rmse_total": total_rmse_from_components(rmse_eta, rmse_u, rmse_v),
        "ke_fd": ke_fd,
        "ke_ml": ke_ml,
    }

def method_label(method_name, nu=0.0, alpha=0.0):
    if method_name == "plain":
        return "plain"
    if method_name == "diffusion":
        return f"diff_nu_{nu:.1e}"
    if method_name == "geonudge":
        return f"geonudge_a_{alpha:.2f}"
    return method_name

# ------------------------------------------------------------
# Run all evaluations
# ------------------------------------------------------------
summary_rows = []
all_results = {}  # all_results[ic_key][run_name][label] = result

for ic_key in selected_cases:
    print(f"\n[Eval] case: {ic_key}")

    truth_pack = load_truth_case(ic_key)
    if truth_pack is None:
        print(f"[Skip] {ic_key}: no usable truth sequence.")
        continue

    all_results[ic_key] = {}

    for rr in loaded_runs:
        run_name = rr["name"]
        model = rr["model"]

        print(f"  [Run] {run_name}")
        all_results[ic_key][run_name] = {}

        method_specs = []
        if RUN_PLAIN:
            method_specs.append(("plain", 0.0, 0.0))
        if RUN_DIFFUSION:
            for nu in NU_LIST:
                method_specs.append(("diffusion", nu, 0.0))
        if RUN_GEONUDGE:
            for alpha in ALPHA_LIST:
                method_specs.append(("geonudge", 0.0, alpha))

        for method_name, nu, alpha in method_specs:
            result = rollout_case(
                model=model,
                truth_pack=truth_pack,
                method_name=method_name,
                nu=nu,
                alpha=alpha
            )

            label = method_label(method_name, nu=nu, alpha=alpha)
            all_results[ic_key][run_name][label] = result

            final_ke_ratio = float(result["ke_ml"][-1] / max(result["ke_fd"][-1], 1e-12))

            summary_rows.append({
                "ic_key": ic_key,
                "run_name": run_name,
                "method": method_name,
                "nu": nu if method_name == "diffusion" else "",
                "alpha": alpha if method_name == "geonudge" else "",
                "rollout_steps": int(truth_pack["rollout_steps"]),
                "final_eta_rmse": float(result["rmse_eta"][-1]),
                "final_u_rmse": float(result["rmse_u"][-1]),
                "final_v_rmse": float(result["rmse_v"][-1]),
                "final_total_rmse": float(result["rmse_total"][-1]),
                "final_fd_ke": float(result["ke_fd"][-1]),
                "final_ml_ke": float(result["ke_ml"][-1]),
                "final_ke_ratio": final_ke_ratio,
            })

            print(
                f"    {label:20s} | "
                f"eta={result['rmse_eta'][-1]:.4f}  "
                f"u={result['rmse_u'][-1]:.4f}  "
                f"v={result['rmse_v'][-1]:.4f}  "
                f"tot={result['rmse_total'][-1]:.4f}  "
                f"KE ratio={final_ke_ratio:.4e}"
            )

# ------------------------------------------------------------
# Save summary CSV
# ------------------------------------------------------------
if len(summary_rows) > 0:
    summary_csv = os.path.join(EVAL_DIR, "summary_eval_compare_3runs.csv")
    keys = list(summary_rows[0].keys())
    with open(summary_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for row in summary_rows:
            w.writerow(row)
    print("\n[Eval] wrote summary CSV:", summary_csv)
else:
    print("\n[Eval] no successful evaluations.")

# ------------------------------------------------------------
# Save best-by-case-and-run table
# ------------------------------------------------------------
if len(summary_rows) > 0:
    best_rows = []
    for ic_key in all_results.keys():
        for run_name in all_results[ic_key].keys():
            candidates = [r for r in summary_rows if r["ic_key"] == ic_key and r["run_name"] == run_name]
            if len(candidates) == 0:
                continue
            best = min(candidates, key=lambda r: r["final_total_rmse"])
            best_rows.append(best)

    if len(best_rows) > 0:
        best_csv = os.path.join(EVAL_DIR, "best_method_per_case_and_run.csv")
        keys = list(best_rows[0].keys())
        with open(best_csv, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            for row in best_rows:
                w.writerow(row)
        print("[Eval] wrote best-method CSV:", best_csv)

# ------------------------------------------------------------
# Plotting helpers
# ------------------------------------------------------------
def save_plot(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)

# ------------------------------------------------------------
# Per-case plots comparing RUNS for the SAME method
# ------------------------------------------------------------
for ic_key, run_pack in all_results.items():
    # gather all method labels present
    labels = []
    for run_name in run_pack.keys():
        for lab in run_pack[run_name].keys():
            if lab not in labels:
                labels.append(lab)

    for lab in labels:
        present_run_names = [rn for rn in run_pack.keys() if lab in run_pack[rn]]
        if len(present_run_names) == 0:
            continue

        sample = run_pack[present_run_names[0]][lab]
        hrs = sample["roll_hours"]

        # Combined RMSE vs time
        fig = plt.figure(figsize=(9, 5))
        for run_name in present_run_names:
            res = run_pack[run_name][lab]
            plt.plot(hrs, res["rmse_total"], marker="o", label=run_name)
        plt.xlabel("Rollout time (hours)")
        plt.ylabel("Combined RMSE")
        plt.title(f"Combined RMSE vs time | {ic_key} | {lab}")
        plt.legend()
        save_plot(fig, os.path.join(EVAL_DIR, f"{ic_key}_{lab}_compare_runs_total_rmse.png"))

        # eta RMSE
        fig = plt.figure(figsize=(9, 5))
        for run_name in present_run_names:
            res = run_pack[run_name][lab]
            plt.plot(hrs, res["rmse_eta"], marker="o", label=run_name)
        plt.xlabel("Rollout time (hours)")
        plt.ylabel("eta RMSE")
        plt.title(f"eta RMSE vs time | {ic_key} | {lab}")
        plt.legend()
        save_plot(fig, os.path.join(EVAL_DIR, f"{ic_key}_{lab}_compare_runs_eta_rmse.png"))

        # u RMSE
        fig = plt.figure(figsize=(9, 5))
        for run_name in present_run_names:
            res = run_pack[run_name][lab]
            plt.plot(hrs, res["rmse_u"], marker="o", label=run_name)
        plt.xlabel("Rollout time (hours)")
        plt.ylabel("u RMSE")
        plt.title(f"u RMSE vs time | {ic_key} | {lab}")
        plt.legend()
        save_plot(fig, os.path.join(EVAL_DIR, f"{ic_key}_{lab}_compare_runs_u_rmse.png"))

        # v RMSE
        fig = plt.figure(figsize=(9, 5))
        for run_name in present_run_names:
            res = run_pack[run_name][lab]
            plt.plot(hrs, res["rmse_v"], marker="o", label=run_name)
        plt.xlabel("Rollout time (hours)")
        plt.ylabel("v RMSE")
        plt.title(f"v RMSE vs time | {ic_key} | {lab}")
        plt.legend()
        save_plot(fig, os.path.join(EVAL_DIR, f"{ic_key}_{lab}_compare_runs_v_rmse.png"))

        # KE vs time
        fig = plt.figure(figsize=(9, 5))
        plt.plot(hrs, sample["ke_fd"], linewidth=2.5, label="FD KE")
        for run_name in present_run_names:
            res = run_pack[run_name][lab]
            plt.plot(hrs, res["ke_ml"], marker="o", label=f"{run_name}")
        plt.xlabel("Rollout time (hours)")
        plt.ylabel("Domain-mean kinetic energy")
        plt.title(f"KE vs time | {ic_key} | {lab}")
        plt.legend()
        save_plot(fig, os.path.join(EVAL_DIR, f"{ic_key}_{lab}_compare_runs_ke.png"))

# ------------------------------------------------------------
# Per-case plots comparing METHODS for the SAME run
# ------------------------------------------------------------
for ic_key, run_pack in all_results.items():
    for run_name, method_pack in run_pack.items():
        labels = list(method_pack.keys())
        if len(labels) == 0:
            continue

        sample = method_pack[labels[0]]
        hrs = sample["roll_hours"]

        # Combined RMSE vs time
        fig = plt.figure(figsize=(10, 5))
        for lab in labels:
            res = method_pack[lab]
            plt.plot(hrs, res["rmse_total"], marker="o", label=lab)
        plt.xlabel("Rollout time (hours)")
        plt.ylabel("Combined RMSE")
        plt.title(f"Combined RMSE vs time | {ic_key} | {run_name}")
        plt.legend(fontsize=8)
        save_plot(fig, os.path.join(EVAL_DIR, f"{ic_key}_{run_name}_compare_methods_total_rmse.png"))

        # KE vs time
        fig = plt.figure(figsize=(10, 5))
        plt.plot(hrs, sample["ke_fd"], linewidth=2.5, label="FD KE")
        for lab in labels:
            res = method_pack[lab]
            plt.plot(hrs, res["ke_ml"], marker="o", label=lab)
        plt.xlabel("Rollout time (hours)")
        plt.ylabel("Domain-mean kinetic energy")
        plt.title(f"KE vs time | {ic_key} | {run_name}")
        plt.legend(fontsize=8)
        save_plot(fig, os.path.join(EVAL_DIR, f"{ic_key}_{run_name}_compare_methods_ke.png"))

# ------------------------------------------------------------
# Cross-case aggregate plots: mean final metrics by run/method
# ------------------------------------------------------------
if len(summary_rows) > 0:
    # collect unique methods in display order
    method_keys = []
    for row in summary_rows:
        lab = method_label(row["method"], row["nu"] if row["nu"] != "" else 0.0,
                           row["alpha"] if row["alpha"] != "" else 0.0)
        if lab not in method_keys:
            method_keys.append(lab)

    # map summary rows by run + label
    agg = {}
    for row in summary_rows:
        run_name = row["run_name"]
        lab = method_label(row["method"], row["nu"] if row["nu"] != "" else 0.0,
                           row["alpha"] if row["alpha"] != "" else 0.0)
        agg.setdefault(run_name, {})
        agg[run_name].setdefault(lab, {
            "final_total_rmse": [],
            "final_ke_ratio": [],
            "final_eta_rmse": [],
            "final_u_rmse": [],
            "final_v_rmse": [],
        })
        agg[run_name][lab]["final_total_rmse"].append(row["final_total_rmse"])
        agg[run_name][lab]["final_ke_ratio"].append(row["final_ke_ratio"])
        agg[run_name][lab]["final_eta_rmse"].append(row["final_eta_rmse"])
        agg[run_name][lab]["final_u_rmse"].append(row["final_u_rmse"])
        agg[run_name][lab]["final_v_rmse"].append(row["final_v_rmse"])

    # write aggregate CSV
    agg_rows = []
    for run_name in agg:
        for lab in agg[run_name]:
            dd = agg[run_name][lab]
            agg_rows.append({
                "run_name": run_name,
                "label": lab,
                "mean_final_total_rmse": float(np.mean(dd["final_total_rmse"])),
                "mean_final_ke_ratio": float(np.mean(dd["final_ke_ratio"])),
                "mean_final_eta_rmse": float(np.mean(dd["final_eta_rmse"])),
                "mean_final_u_rmse": float(np.mean(dd["final_u_rmse"])),
                "mean_final_v_rmse": float(np.mean(dd["final_v_rmse"])),
            })

    agg_csv = os.path.join(EVAL_DIR, "aggregate_mean_metrics_by_run_and_method.csv")
    with open(agg_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(agg_rows[0].keys()))
        w.writeheader()
        for row in agg_rows:
            w.writerow(row)
    print("[Eval] wrote aggregate CSV:", agg_csv)

    # mean final total RMSE bar chart
    fig = plt.figure(figsize=(12, 6))
    x_labels = []
    y_vals = []
    for row in agg_rows:
        x_labels.append(f"{row['run_name']}\n{row['label']}")
        y_vals.append(row["mean_final_total_rmse"])
    plt.bar(np.arange(len(y_vals)), y_vals)
    plt.xticks(np.arange(len(y_vals)), x_labels, rotation=90)
    plt.ylabel("Mean final total RMSE")
    plt.title("Mean final total RMSE across selected cases")
    save_plot(fig, os.path.join(EVAL_DIR, "aggregate_mean_final_total_rmse.png"))

    # mean final KE ratio bar chart
    fig = plt.figure(figsize=(12, 6))
    x_labels = []
    y_vals = []
    for row in agg_rows:
        x_labels.append(f"{row['run_name']}\n{row['label']}")
        y_vals.append(row["mean_final_ke_ratio"])
    plt.bar(np.arange(len(y_vals)), y_vals)
    plt.xticks(np.arange(len(y_vals)), x_labels, rotation=90)
    plt.ylabel("Mean final KE ratio")
    plt.title("Mean final KE ratio across selected cases")
    save_plot(fig, os.path.join(EVAL_DIR, "aggregate_mean_final_ke_ratio.png"))

# ------------------------------------------------------------
# Optional field plots
# ------------------------------------------------------------
if MAKE_FIELD_PLOTS:
    for ic_key, run_pack in all_results.items():
        if FIELD_PLOT_RUN_NAME not in run_pack:
            continue
        if FIELD_PLOT_METHOD not in run_pack[FIELD_PLOT_RUN_NAME]:
            continue

        result = run_pack[FIELD_PLOT_RUN_NAME][FIELD_PLOT_METHOD]
        truth_seq = result["truth_seq"]
        pred_seq = result["pred_seq"]
        rollout_steps = len(result["rmse_eta"])

        for kk in FIELD_PLOT_STEPS:
            if kk < 0 or kk >= rollout_steps:
                continue

            t_idx = kk + 1

            eta_fd = truth_seq[t_idx, 0]
            u_fd   = truth_seq[t_idx, 1]
            v_fd   = truth_seq[t_idx, 2]

            eta_ml = pred_seq[t_idx, 0]
            u_ml   = pred_seq[t_idx, 1]
            v_ml   = pred_seq[t_idx, 2]

            fig, axes = plt.subplots(3, 3, figsize=(14, 11))

            im = axes[0, 0].imshow(eta_fd, origin="lower")
            axes[0, 0].set_title(f"FD eta | step {t_idx}")
            plt.colorbar(im, ax=axes[0, 0], fraction=0.046, pad=0.04)

            im = axes[0, 1].imshow(eta_ml, origin="lower")
            axes[0, 1].set_title(f"{FIELD_PLOT_RUN_NAME} {FIELD_PLOT_METHOD} eta")
            plt.colorbar(im, ax=axes[0, 1], fraction=0.046, pad=0.04)

            im = axes[0, 2].imshow(eta_ml - eta_fd, origin="lower")
            axes[0, 2].set_title("ML - FD eta")
            plt.colorbar(im, ax=axes[0, 2], fraction=0.046, pad=0.04)

            im = axes[1, 0].imshow(u_fd, origin="lower")
            axes[1, 0].set_title("FD u")
            plt.colorbar(im, ax=axes[1, 0], fraction=0.046, pad=0.04)

            im = axes[1, 1].imshow(u_ml, origin="lower")
            axes[1, 1].set_title("ML u")
            plt.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)

            im = axes[1, 2].imshow(u_ml - u_fd, origin="lower")
            axes[1, 2].set_title("ML - FD u")
            plt.colorbar(im, ax=axes[1, 2], fraction=0.046, pad=0.04)

            im = axes[2, 0].imshow(v_fd, origin="lower")
            axes[2, 0].set_title("FD v")
            plt.colorbar(im, ax=axes[2, 0], fraction=0.046, pad=0.04)

            im = axes[2, 1].imshow(v_ml, origin="lower")
            axes[2, 1].set_title("ML v")
            plt.colorbar(im, ax=axes[2, 1], fraction=0.046, pad=0.04)

            im = axes[2, 2].imshow(v_ml - v_fd, origin="lower")
            axes[2, 2].set_title("ML - FD v")
            plt.colorbar(im, ax=axes[2, 2], fraction=0.046, pad=0.04)

            plt.tight_layout()
            out_name = f"{ic_key}_{FIELD_PLOT_RUN_NAME}_{FIELD_PLOT_METHOD}_fields_step_{t_idx:03d}.png"
            plt.savefig(os.path.join(EVAL_DIR, out_name), dpi=150)
            plt.close()

print("\n[Eval] finished.")
print("[Eval] outputs saved in:", EVAL_DIR)

Using device: cuda
GPU: Tesla T4
[Load] b8_c96_t4_p4 <- /content/drive/MyDrive/AI_4DVAR2/training_runs/rescnn_b8_c96_lr1e4_t4_p4/final_model.pt
[Load] b8_c96_t4_p8 <- /content/drive/MyDrive/AI_4DVAR2/training_runs/rescnn_b8_c96_lr1e4_t4_p8/final_model.pt
[Load] b8_c96_t6_p4 <- /content/drive/MyDrive/AI_4DVAR2/training_runs/rescnn_b8_c96_lr1e4_t6_p4/final_model.pt
[Eval] using FD data dir: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
[Eval] found 28 ICs
[Eval] available ICs: ['gauss_00', 'gauss_01', 'gauss_02', 'gauss_03', 'gauss_04', 'gauss_05', 'mix_00', 'mix_01', 'mix_02', 'mix_03', 'mix_04', 'mix_05', 'test_modon_00', 'test_modon_01', 'test_rh_00', 'test_rh_01', 'vort_00', 'vort_01', 'vort_02', 'vort_03', 'vort_04', 'vort_05', 'wave_00', 'wave_01', 'wave_02', 'wave_03', 'wave_04', 'wave_05']
[Eval] selected cases: ['gauss_00', 'test_modon_00', 'test_rh_00', 'wave_00']

[Eval] case: gauss_00
  [Run] b8_c96_t4_p4
    plain                | eta=12.2192  u=6.9803  v=0.8176  tot=14.0962

# Modified residual CNN with KE constraint

In [6]:
# ============================================================
# train_rescnn_1layer_configurable_ai4dvar2_keweak.py
# ------------------------------------------------------------
# Configurable residual-CNN trainer for 1-layer SWE emulator
# on Klein beta-plane, with:
#   - multistep rollout data loss
#   - collocation state loss
#   - collocation tendency loss
#   - continuity residual (flux form)
#   - momentum residual (nonlinear)
#   - weak geostrophic-balance loss
#   - weak damping / smoothness loss
#   - NEW: weak domain-mean KE constraint
#
# Outputs go to:
#   /content/drive/MyDrive/AI_4DVAR2/...
# ============================================================

import os
import sys
import glob
import csv
import time
import random
import re
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# Import CollocBank
# ------------------------------------------------------------
sys.path.append("/content/drive/MyDrive/AI_4DVAR")
from colloc_bank import CollocBank

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# USER CONFIG
# ------------------------------------------------------------
class CFG:
    # -------- paths --------
    ROOT_IN  = "/content/drive/MyDrive/AI_4DVAR"
    ROOT_OUT = "/content/drive/MyDrive/AI_4DVAR2"

    DATA_DIR_CANDIDATES = [
        f"{ROOT_IN}/klein_ckpt_1L",
        f"{ROOT_IN}/klein_ckpt_AL_centers_colloc",
        f"{ROOT_IN}/klein_ckpt_AL_centers",
        f"{ROOT_IN}/klein_ckpt_1L_colloc",
    ]
    COLLOC_DIR = f"{ROOT_IN}/klein_ml_1L/colloc_raw"

    # customize this tag per experiment
    # EXP_NAME = "rescnn_b8_c96_lr1e4_t6_p4_keweak"
    # EXP_NAME = "rescnn_b12_c96_lr1e4_t6_p4_keweak"
    # EXP_NAME = "rescnn_b12_c96_lr1e4_t8_p4_keweak"
    # EXP_NAME = "rescnn_b12_c96_lr1e4_t8_p4_keweak_lam001"
    # EXP_NAME = "rescnn_b12_c128_lr1e4_t8_p4_keweak"
    # EXP_NAME = "rescnn_b12_c96_lr1e4_t6_p4_keweak_lam001"
    EXP_NAME = "rescnn_b12_c128_lr1e4_t6_p4_keweak"

    # -------- domain / physics --------
    NX = 256
    NY = 128
    Lx = 2.0e7
    Ly = 8.0e6
    H = 1000.0
    G = 9.81
    DT_MACRO = 200.0 * 30.0
    FMIN = 2.0e-5

    # -------- architecture --------
    # N_BLOCKS = 8
    N_BLOCKS = 12
    # FEAT_CH  = 96
    # HIDDEN   = 192
    FEAT_CH  = 128
    HIDDEN   = 256

    # -------- optimization --------
    EPOCHS = 12
    BATCH_SIZE = 1
    LR = 1e-4
    WEIGHT_DECAY = 1e-6
    GRAD_CLIP = 1.0

    # -------- rollout --------
    ROLL_STEPS = 4
    ROLL_GAMMA = 0.95

    # -------- data loading --------
    MAX_WINDOWS_PER_IC = 4
    NUM_WORKERS = 0
    PIN_MEMORY = torch.cuda.is_available()

    # -------- collocation controls --------
    # N_TIME_SLICES = 8
    N_TIME_SLICES = 6
    PTS_PER_TIME  = 4

    # -------- loss weights --------
    LAMBDA_DATA       = 1.0
    LAMBDA_COLL_STATE = 0.05
    LAMBDA_COLL_TEND  = 0.1
    LAMBDA_CONT       = 0.2
    LAMBDA_MOM        = 0.5
    LAMBDA_GEO        = 0.01
    LAMBDA_SMOOTH     = 1e-4

    # -------- NEW: weak KE loss --------
    # Uses log-domain mismatch of domain-mean KE over rollout.
    # Start weak. If too weak, try 0.05; if unstable, try 0.005.
    LAMBDA_KE = 0.02
    # LAMBDA_KE = 0.01
    KE_EPS = 1e-6

    # -------- misc --------
    SAVE_EVERY_EPOCH = 1
    PRINT_BATCH_EVERY = 10
    RESUME_FROM = None

cfg = CFG()

cfg.DX = cfg.Lx / cfg.NX
cfg.DY = cfg.Ly / cfg.NY
cfg.CKPT_DIR = os.path.join(cfg.ROOT_OUT, "training_runs", cfg.EXP_NAME)
os.makedirs(cfg.CKPT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0
    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic = 0
    n_pairs = 0
    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += (len(files) - 1)
    return n_ic, n_pairs

def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir = None
    best_pairs = -1
    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_pairs = n_pairs
            best_dir = d
    if best_dir is None or best_pairs <= 0:
        raise RuntimeError("No valid snapshot root found.")
    print(f"[AutoDetect] using: {best_dir}")
    return best_dir

_step_re = re.compile(r"klein_step_(\d+)\.npz")

def extract_step_from_path(path):
    m = _step_re.search(os.path.basename(path))
    if m is None:
        raise ValueError(f"Could not parse step from {path}")
    return int(m.group(1))

def save_checkpoint(path, model, optimizer, epoch, loss_history, data_dir):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss_history": loss_history,
            "data_dir": data_dir,
            "exp_name": cfg.EXP_NAME,
            "config": {k: v for k, v in cfg.__dict__.items() if not k.startswith("__")},
        },
        path,
    )

def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    return ckpt.get("epoch", -1), ckpt.get("loss_history", []), ckpt

def save_loss_csv(path, loss_history):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow([
            "epoch", "train_total", "train_data",
            "train_coll_state", "train_coll_tend",
            "train_cont", "train_mom",
            "train_geo", "train_smooth",
            "train_ke"
        ])
        for row in loss_history:
            w.writerow(row)

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class SWWindowDataset(Dataset):
    def __init__(self, data_dir, roll_steps=4, max_windows_per_ic=None):
        self.samples = []

        ic_dirs = sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)])
        print(f"[Dataset] found {len(ic_dirs)} IC directories")

        for ic_dir in ic_dirs:
            ic_key = os.path.basename(ic_dir)
            files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
            print(f"[Dataset] {ic_key:20s} -> {len(files)} snapshot files")

            if len(files) < (roll_steps + 1):
                continue

            windows = []
            for i in range(len(files) - roll_steps):
                fseq = files[i:i + roll_steps + 1]
                macro_start_index = i + 1
                windows.append((fseq, ic_key, macro_start_index))

            if max_windows_per_ic is not None:
                windows = windows[:max_windows_per_ic]

            self.samples.extend(windows)

        if len(self.samples) == 0:
            raise RuntimeError("No usable training windows found.")

        print(f"[Dataset] total windows = {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fseq, ic_key, macro_start_index = self.samples[idx]

        seq = []
        times = []
        steps = []

        for f in fseq:
            z = np.load(f)
            seq.append(np.stack([z["eta"], z["uc"], z["vc"]], axis=0).astype(np.float32))
            times.append(float(z["t"]))
            steps.append(int(extract_step_from_path(f)))

        seq = np.stack(seq, axis=0)

        return {
            "seq": torch.from_numpy(seq),
            "times": np.array(times, dtype=np.float32),
            "steps": np.array(steps, dtype=np.int32),
            "ic_key": ic_key,
            "macro_start_index": np.int32(macro_start_index),
        }

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
class ResBlock(nn.Module):
    def __init__(self, ch, dilation=1):
        super().__init__()
        pad = dilation
        self.c1 = nn.Conv2d(ch, ch, 3, padding=pad, dilation=dilation)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=pad, dilation=dilation)
        self.act = nn.GELU()

    def forward(self, x):
        r = x
        x = self.act(self.c1(x))
        x = self.c2(x)
        return self.act(x + r)

class MultiStepContinuousResCNNModel(nn.Module):
    def __init__(self, in_ch=3, feat_ch=96, hidden=192, n_blocks=8):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
        )

        blocks = []
        dilations = [1, 1, 2, 2, 1, 1, 2, 2, 1, 1, 2, 2]
        for i in range(n_blocks):
            blocks.append(ResBlock(feat_ch, dilation=dilations[i % len(dilations)]))
        self.resnet = nn.Sequential(*blocks)

        self.grid_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, 3, 3, padding=1),
        )

        self.query_mlp = nn.Sequential(
            nn.Linear(feat_ch + 3 + 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

        # zero-init safe start
        nn.init.zeros_(self.grid_head[-1].weight)
        nn.init.zeros_(self.grid_head[-1].bias)
        nn.init.zeros_(self.query_mlp[-1].weight)
        nn.init.zeros_(self.query_mlp[-1].bias)

    def encode(self, x):
        return self.resnet(self.stem(x))

    def grid_tendency(self, feat):
        return self.grid_head(feat)

    def _sample_local(self, field_1b, x_norm_detached, y_norm_detached):
        grid = torch.stack([x_norm_detached, y_norm_detached], dim=-1).view(1, -1, 1, 2)
        vals = F.grid_sample(
            field_1b, grid,
            mode="bilinear",
            padding_mode="border",
            align_corners=True,
        )
        return vals.squeeze(0).squeeze(-1).transpose(0, 1)

    def query_state(self, x_1b, feat_1b, x_norm, y_norm, tau):
        x_norm_s = x_norm.detach()
        y_norm_s = y_norm.detach()

        feat_local = self._sample_local(feat_1b, x_norm_s, y_norm_s)
        state0_local = self._sample_local(x_1b, x_norm_s, y_norm_s)

        q = torch.stack([x_norm, y_norm, tau], dim=-1)
        inp = torch.cat([feat_local, state0_local, q], dim=-1)

        delta = self.query_mlp(inp)
        return state0_local + tau.unsqueeze(-1) * delta

# ------------------------------------------------------------
# Collocation sampling
# ------------------------------------------------------------
def sample_multitime_colloc(colloc_bank, ic_key, macro_index, n_time_slices=4, pts_per_time=4):
    n_request = max(n_time_slices * pts_per_time * 8, 64)
    base = colloc_bank.sample_nearest_macro(
        ic_key=ic_key,
        macro_index=macro_index,
        npts=n_request,
        replace=True,
    )

    t_sec = np.asarray(base["t_sec"]).reshape(-1)
    if t_sec.size == 0:
        return base

    order = np.argsort(t_sec)
    bins = np.array_split(order, n_time_slices)

    chosen = []
    for b in bins:
        if len(b) == 0:
            continue
        take = min(pts_per_time, len(b))
        idx = np.random.choice(b, size=take, replace=False)
        chosen.append(idx)

    if len(chosen) == 0:
        chosen = [order[:min(pts_per_time, len(order))]]

    chosen = np.concatenate(chosen, axis=0)

    out = {}
    for k, arr in base.items():
        arr = np.asarray(arr)
        if arr.ndim >= 1 and arr.shape[0] == len(t_sec):
            out[k] = arr[chosen]
        else:
            out[k] = arr
    return out

# ------------------------------------------------------------
# Derivative / balance helpers
# ------------------------------------------------------------
def safe_coriolis_torch(f, fmin):
    sign = torch.sign(f)
    sign = torch.where(sign == 0.0, torch.ones_like(sign), sign)
    return sign * torch.clamp(torch.abs(f), min=fmin)

# ------------------------------------------------------------
# NEW: domain-mean KE helper
# ------------------------------------------------------------
def domain_mean_ke_torch(state, h_floor=1.0):
    """
    state: [B, 3, NY, NX] with channels [eta, u, v]
    returns: [B] domain-mean KE
    """
    eta = state[:, 0]
    u   = state[:, 1]
    v   = state[:, 2]

    h = cfg.H + eta
    h_safe = torch.clamp(h, min=h_floor)

    ke = 0.5 * h_safe * (u * u + v * v)
    return ke.mean(dim=(-2, -1))

# ------------------------------------------------------------
# Collocation losses
# ------------------------------------------------------------
def continuous_interval_losses_multitime(model, x_1b, feat_1b, colloc, t_start, dt_macro):
    x_m = torch.as_tensor(colloc["x_m"], dtype=torch.float32, device=device)
    y_m = torch.as_tensor(colloc["y_m"], dtype=torch.float32, device=device)
    t_sec = torch.as_tensor(colloc["t_sec"], dtype=torch.float32, device=device)

    tau = (t_sec - t_start) / dt_macro
    tau = torch.clamp(tau, 0.0, 1.0)

    x_norm = (2.0 * x_m / cfg.Lx) - 1.0
    y_norm = (2.0 * y_m / cfg.Ly) - 1.0

    x_norm = x_norm.clone().detach().requires_grad_(True)
    y_norm = y_norm.clone().detach().requires_grad_(True)
    tau    = tau.clone().detach().requires_grad_(True)

    state_hat = model.query_state(x_1b, feat_1b, x_norm, y_norm, tau)
    eta_hat = state_hat[:, 0]
    u_hat   = state_hat[:, 1]
    v_hat   = state_hat[:, 2]
    h_hat   = cfg.H + eta_hat

    def grads_of(field):
        gx, gy, gt = torch.autograd.grad(
            field.sum(), [x_norm, y_norm, tau],
            create_graph=True, retain_graph=True
        )
        return gx, gy, gt

    eta_xn, eta_yn, eta_tau = grads_of(eta_hat)
    u_xn,   u_yn,   u_tau   = grads_of(u_hat)
    v_xn,   v_yn,   v_tau   = grads_of(v_hat)

    eta_x = eta_xn * (2.0 / cfg.Lx)
    eta_y = eta_yn * (2.0 / cfg.Ly)
    h_x = eta_x
    h_y = eta_y

    u_x = u_xn * (2.0 / cfg.Lx)
    u_y = u_yn * (2.0 / cfg.Ly)
    v_x = v_xn * (2.0 / cfg.Lx)
    v_y = v_yn * (2.0 / cfg.Ly)

    eta_t = eta_tau / dt_macro
    h_t = eta_t
    u_t = u_tau / dt_macro
    v_t = v_tau / dt_macro

    hu = h_hat * u_hat
    hv = h_hat * v_hat
    hu_xn, _, _ = grads_of(hu)
    _, hv_yn, _ = grads_of(hv)
    hu_x = hu_xn * (2.0 / cfg.Lx)
    hv_y = hv_yn * (2.0 / cfg.Ly)

    eta_true = torch.as_tensor(colloc["eta"], dtype=torch.float32, device=device)
    u_true   = torch.as_tensor(colloc["uc"],  dtype=torch.float32, device=device)
    v_true   = torch.as_tensor(colloc["vc"],  dtype=torch.float32, device=device)

    deta_dt_fd = torch.as_tensor(colloc["deta_dt_fd"], dtype=torch.float32, device=device)
    duc_dt_fd  = torch.as_tensor(colloc["duc_dt_fd"],  dtype=torch.float32, device=device)
    dvc_dt_fd  = torch.as_tensor(colloc["dvc_dt_fd"],  dtype=torch.float32, device=device)

    f_fd = torch.as_tensor(colloc["f"], dtype=torch.float32, device=device)

    # state collocation
    loss_coll_state = ((eta_hat - eta_true) ** 2).mean()
    loss_coll_state += ((u_hat - u_true) ** 2).mean()
    loss_coll_state += ((v_hat - v_true) ** 2).mean()

    # tendency collocation
    loss_coll_tend = ((eta_t - deta_dt_fd) ** 2).mean()
    loss_coll_tend += ((u_t - duc_dt_fd) ** 2).mean()
    loss_coll_tend += ((v_t - dvc_dt_fd) ** 2).mean()

    # continuity in flux form
    resid_cont = h_t + hu_x + hv_y
    loss_cont = (resid_cont ** 2).mean()

    # nonlinear momentum
    adv_u = u_hat * u_x + v_hat * u_y
    adv_v = u_hat * v_x + v_hat * v_y
    resid_u = u_t + adv_u - f_fd * v_hat + cfg.G * h_x
    resid_v = v_t + adv_v + f_fd * u_hat + cfg.G * h_y
    loss_mom = (resid_u ** 2).mean() + (resid_v ** 2).mean()

    # weak geostrophic balance penalty
    f_safe = safe_coriolis_torch(f_fd, cfg.FMIN)
    u_g = -(cfg.G / f_safe) * eta_y
    v_g =  (cfg.G / f_safe) * eta_x
    loss_geo = ((u_hat - u_g) ** 2).mean() + ((v_hat - v_g) ** 2).mean()

    # weak smoothness / damping penalty
    loss_smooth = (u_x ** 2 + u_y ** 2 + v_x ** 2 + v_y ** 2).mean()

    return loss_coll_state, loss_coll_tend, loss_cont, loss_mom, loss_geo, loss_smooth

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
def train():
    data_dir = autodetect_data_dir(cfg.DATA_DIR_CANDIDATES)

    if not os.path.exists(cfg.COLLOC_DIR):
        raise RuntimeError(f"COLLOC_DIR does not exist: {cfg.COLLOC_DIR}")

    colloc_bank = CollocBank(cfg.COLLOC_DIR, verbose=True)

    dataset = SWWindowDataset(
        data_dir,
        roll_steps=cfg.ROLL_STEPS,
        max_windows_per_ic=cfg.MAX_WINDOWS_PER_IC,
    )

    loader = DataLoader(
        dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=cfg.PIN_MEMORY,
        drop_last=False,
    )

    model = MultiStepContinuousResCNNModel(
        feat_ch=cfg.FEAT_CH,
        hidden=cfg.HIDDEN,
        n_blocks=cfg.N_BLOCKS,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)

    start_epoch = 0
    loss_history = []

    if cfg.RESUME_FROM is not None and os.path.exists(cfg.RESUME_FROM):
        print(f"[Resume] loading checkpoint: {cfg.RESUME_FROM}")
        loaded_epoch, loaded_history, _ = load_checkpoint(cfg.RESUME_FROM, model, optimizer)
        start_epoch = loaded_epoch + 1
        loss_history = loaded_history

    print(f"[Train] output dir            = {cfg.CKPT_DIR}")
    print(f"[Train] windows               = {len(dataset)}")
    print(f"[Train] batches/epoch         = {len(loader)}")
    print(f"[Train] N_BLOCKS              = {cfg.N_BLOCKS}")
    print(f"[Train] FEAT_CH               = {cfg.FEAT_CH}")
    print(f"[Train] LR                    = {cfg.LR}")
    print(f"[Train] N_TIME_SLICES         = {cfg.N_TIME_SLICES}")
    print(f"[Train] PTS_PER_TIME          = {cfg.PTS_PER_TIME}")
    print(f"[Train] colloc/interval       = {cfg.N_TIME_SLICES * cfg.PTS_PER_TIME}")
    print(f"[Train] LAMBDA_KE             = {cfg.LAMBDA_KE}")

    for epoch in range(start_epoch, cfg.EPOCHS):
        t0 = time.time()
        model.train()

        run_total = 0.0
        run_data = 0.0
        run_cstate = 0.0
        run_ctend = 0.0
        run_cont = 0.0
        run_mom = 0.0
        run_geo = 0.0
        run_smooth = 0.0
        run_ke = 0.0

        for ib, batch in enumerate(loader):
            seq = batch["seq"].to(device, non_blocking=True)
            times = batch["times"].numpy()
            ic_keys = batch["ic_key"]
            macro_start = batch["macro_start_index"].numpy()

            B = seq.shape[0]
            optimizer.zero_grad(set_to_none=True)

            loss_data = torch.tensor(0.0, device=device)
            loss_cstate = torch.tensor(0.0, device=device)
            loss_ctend = torch.tensor(0.0, device=device)
            loss_cont = torch.tensor(0.0, device=device)
            loss_mom = torch.tensor(0.0, device=device)
            loss_geo = torch.tensor(0.0, device=device)
            loss_smooth = torch.tensor(0.0, device=device)

            # NEW: KE loss
            loss_ke = torch.tensor(0.0, device=device)

            n_phys = 0

            x = seq[:, 0]

            for k in range(cfg.ROLL_STEPS):
                feat = model.encode(x)
                xdot_grid = model.grid_tendency(feat)
                x_next = x + cfg.DT_MACRO * xdot_grid

                truth_next = seq[:, k + 1]
                wk = (cfg.ROLL_GAMMA ** k)

                # main state loss
                loss_data = loss_data + wk * ((x_next - truth_next) ** 2).mean()

                # NEW: weak KE constraint in log space
                ke_pred = domain_mean_ke_torch(x_next)
                ke_true = domain_mean_ke_torch(truth_next)

                log_ke_pred = torch.log(ke_pred + cfg.KE_EPS)
                log_ke_true = torch.log(ke_true + cfg.KE_EPS)

                loss_ke = loss_ke + wk * ((log_ke_pred - log_ke_true) ** 2).mean()

                for b in range(B):
                    ic_key = ic_keys[b]
                    macro_idx = int(macro_start[b] + k)
                    t_start = float(times[b, k])

                    if not colloc_bank.has_ic(ic_key):
                        continue

                    colloc = sample_multitime_colloc(
                        colloc_bank=colloc_bank,
                        ic_key=ic_key,
                        macro_index=macro_idx,
                        n_time_slices=cfg.N_TIME_SLICES,
                        pts_per_time=cfg.PTS_PER_TIME,
                    )

                    l_cstate, l_ctend, l_cont, l_mom, l_geo, l_smooth = continuous_interval_losses_multitime(
                        model=model,
                        x_1b=x[b:b+1],
                        feat_1b=feat[b:b+1],
                        colloc=colloc,
                        t_start=t_start,
                        dt_macro=cfg.DT_MACRO,
                    )

                    loss_cstate += l_cstate
                    loss_ctend += l_ctend
                    loss_cont += l_cont
                    loss_mom += l_mom
                    loss_geo += l_geo
                    loss_smooth += l_smooth
                    n_phys += 1

                x = x_next

            if n_phys > 0:
                loss_cstate = loss_cstate / n_phys
                loss_ctend = loss_ctend / n_phys
                loss_cont = loss_cont / n_phys
                loss_mom = loss_mom / n_phys
                loss_geo = loss_geo / n_phys
                loss_smooth = loss_smooth / n_phys
            else:
                zero = torch.tensor(0.0, device=device)
                loss_cstate = zero
                loss_ctend = zero
                loss_cont = zero
                loss_mom = zero
                loss_geo = zero
                loss_smooth = zero

            loss = (
                cfg.LAMBDA_DATA         * loss_data
                + cfg.LAMBDA_COLL_STATE * loss_cstate
                + cfg.LAMBDA_COLL_TEND  * loss_ctend
                + cfg.LAMBDA_CONT       * loss_cont
                + cfg.LAMBDA_MOM        * loss_mom
                + cfg.LAMBDA_GEO        * loss_geo
                + cfg.LAMBDA_SMOOTH     * loss_smooth
                + cfg.LAMBDA_KE         * loss_ke
            )

            if not torch.isfinite(loss):
                print("Non-finite diagnostics:")
                print("  data   =", float(loss_data.detach().cpu()))
                print("  cstate =", float(loss_cstate.detach().cpu()))
                print("  ctend  =", float(loss_ctend.detach().cpu()))
                print("  cont   =", float(loss_cont.detach().cpu()))
                print("  mom    =", float(loss_mom.detach().cpu()))
                print("  geo    =", float(loss_geo.detach().cpu()))
                print("  smooth =", float(loss_smooth.detach().cpu()))
                print("  ke     =", float(loss_ke.detach().cpu()))
                raise RuntimeError(f"Non-finite loss at epoch={epoch}, batch={ib}: {loss.item()}")

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            optimizer.step()

            run_total += float(loss.detach().cpu())
            run_data += float(loss_data.detach().cpu())
            run_cstate += float(loss_cstate.detach().cpu())
            run_ctend += float(loss_ctend.detach().cpu())
            run_cont += float(loss_cont.detach().cpu())
            run_mom += float(loss_mom.detach().cpu())
            run_geo += float(loss_geo.detach().cpu())
            run_smooth += float(loss_smooth.detach().cpu())
            run_ke += float(loss_ke.detach().cpu())

            if (ib % cfg.PRINT_BATCH_EVERY) == 0:
                print(
                    f"[epoch {epoch:03d} | batch {ib:04d}/{len(loader):04d}] "
                    f"total={float(loss.detach().cpu()):.6e} "
                    f"data={float(loss_data.detach().cpu()):.6e} "
                    f"cstate={float(loss_cstate.detach().cpu()):.6e} "
                    f"ctend={float(loss_ctend.detach().cpu()):.6e} "
                    f"cont={float(loss_cont.detach().cpu()):.6e} "
                    f"mom={float(loss_mom.detach().cpu()):.6e} "
                    f"geo={float(loss_geo.detach().cpu()):.6e} "
                    f"smooth={float(loss_smooth.detach().cpu()):.6e} "
                    f"ke={float(loss_ke.detach().cpu()):.6e}"
                )

        n_batches = max(len(loader), 1)
        ep_total = run_total / n_batches
        ep_data = run_data / n_batches
        ep_cstate = run_cstate / n_batches
        ep_ctend = run_ctend / n_batches
        ep_cont = run_cont / n_batches
        ep_mom = run_mom / n_batches
        ep_geo = run_geo / n_batches
        ep_smooth = run_smooth / n_batches
        ep_ke = run_ke / n_batches

        loss_history.append([
            epoch, ep_total, ep_data, ep_cstate, ep_ctend,
            ep_cont, ep_mom, ep_geo, ep_smooth, ep_ke
        ])

        print(
            f"Epoch {epoch:03d} done | "
            f"total={ep_total:.6e} | data={ep_data:.6e} | "
            f"cstate={ep_cstate:.6e} | ctend={ep_ctend:.6e} | "
            f"cont={ep_cont:.6e} | mom={ep_mom:.6e} | "
            f"geo={ep_geo:.6e} | smooth={ep_smooth:.6e} | "
            f"ke={ep_ke:.6e} | "
            f"time={time.time() - t0:.1f}s"
        )

        last_ckpt = os.path.join(cfg.CKPT_DIR, "last_model.pt")
        save_checkpoint(last_ckpt, model, optimizer, epoch, loss_history, data_dir)

        if ((epoch + 1) % cfg.SAVE_EVERY_EPOCH) == 0:
            save_checkpoint(
                os.path.join(cfg.CKPT_DIR, f"model_epoch_{epoch:03d}.pt"),
                model, optimizer, epoch, loss_history, data_dir
            )

        save_loss_csv(os.path.join(cfg.CKPT_DIR, "loss_history.csv"), loss_history)

    final_ckpt = os.path.join(cfg.CKPT_DIR, "final_model.pt")
    save_checkpoint(final_ckpt, model, optimizer, cfg.EPOCHS - 1, loss_history, data_dir)
    save_loss_csv(os.path.join(cfg.CKPT_DIR, "loss_history.csv"), loss_history)

    print("[Train] finished.")
    print(f"[Train] final checkpoint: {final_ckpt}")

if __name__ == "__main__":
    train()

Using device: cuda
GPU: Tesla T4
[AutoDetect] checking candidate snapshot roots...
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
     valid IC dirs = 28, total pairs = 168
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers_colloc
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L_colloc
     valid IC dirs = 0, total pairs = 0
[AutoDetect] using: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
[CollocBank] loaded gauss_00 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_01 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_02 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_03 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_04 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_05 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_00 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_01 | sampl

# Evaluation

In [2]:
# ============================================================
# eval_rescnn_compare_t6_keweak_runs.py
# ------------------------------------------------------------
# Compare these 1-layer ResCNN runs:
#   - b8_c96_t6_p4
#   - b8_c96_t6_p4_keweak
#   - b12_c96_t6_p4_keweak
#
# Features:
#   - exact residual CNN architecture matching training/eval
#   - plain rollout
#   - rollout + u,v diffusion
#   - rollout + weak geostrophic nudging
#   - summary CSV
#   - per-case RMSE vs time plots
#   - per-case KE vs time plots
#   - aggregate mean-metric CSV/plots
#
# Output:
#   /content/drive/MyDrive/AI_4DVAR2/eval_rescnn_compare_t6_keweak_runs
# ============================================================

import os
import re
import glob
import csv
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
ROOT_IN  = "/content/drive/MyDrive/AI_4DVAR"
ROOT_OUT = "/content/drive/MyDrive/AI_4DVAR2"

DATA_DIR_CANDIDATES = [
    f"{ROOT_IN}/klein_ckpt_1L",
    f"{ROOT_IN}/klein_ckpt_AL_centers_colloc",
    f"{ROOT_IN}/klein_ckpt_AL_centers",
    f"{ROOT_IN}/klein_ckpt_1L_colloc",
]

RUNS = [
#    {
#        "name": "b8_c96_t6_p4",
#        "ckpt_dir": f"{ROOT_OUT}/training_runs/rescnn_b8_c96_lr1e4_t6_p4",
#        "ckpt_name": "final_model.pt",
#    },
#    {
#        "name": "b8_c96_t6_p4_keweak",
#        "ckpt_dir": f"{ROOT_OUT}/training_runs/rescnn_b8_c96_lr1e4_t6_p4_keweak",
#        "ckpt_name": "final_model.pt",
#    },
    {
        "name": "b12_c96_t6_p4_keweak",
        "ckpt_dir": f"{ROOT_OUT}/training_runs/rescnn_b12_c96_lr1e4_t6_p4_keweak",
        "ckpt_name": "final_model.pt",
    },
#        {
#        "name": "b12_c96_t8_p4_keweak",
#        "ckpt_dir": f"{ROOT_OUT}/training_runs/rescnn_b12_c96_lr1e4_t8_p4_keweak",
#        "ckpt_name": "final_model.pt",
#    },
        {
        "name": "b12_c96_t6_p4_keweak_lam001",
        "ckpt_dir": f"{ROOT_OUT}/training_runs/rescnn_b12_c96_lr1e4_t6_p4_keweak_lam001",
        "ckpt_name": "final_model.pt",
    },
        {
        "name": "b12_c128_t6_p4_keweak",
        "ckpt_dir": f"{ROOT_OUT}/training_runs/rescnn_b12_c128_lr1e4_t6_p4_keweak",
        "ckpt_name": "final_model.pt",
    },
]

EVAL_DIR = os.path.join(ROOT_OUT, "eval_rescnn_compare_t6_keweak_runs")
os.makedirs(EVAL_DIR, exist_ok=True)

# ------------------------------------------------------------
# Constants
# ------------------------------------------------------------
H = 1000.0
G = 9.81
DT_MACRO = 200.0 * 30.0

Lx = 2.0e7
Ly = 8.0e6
NX = 256
NY = 128
DX = Lx / NX
DY = Ly / NY

FMIN = 2.0e-5

# ------------------------------------------------------------
# USER CONTROLS
# ------------------------------------------------------------
# MODE: "one", "list", "all"
MODE = "list"

ONE_CASE = "test_rh_00"

CASE_LIST = [
    "gauss_00",
    "test_modon_00",
    "test_rh_00",
    "wave_00",
]

START_INDEX = 0
ROLLOUT_STEPS = 6

MAKE_FIELD_PLOTS = False
FIELD_PLOT_RUN_NAME = "b8_c96_t6_p4_keweak"
FIELD_PLOT_METHOD = "plain"
FIELD_PLOT_STEPS = [0, 2, 5]  # rollout indices

RUN_PLAIN = True
RUN_DIFFUSION = True
RUN_GEONUDGE = True

NU_LIST = [5.0e4, 1.0e5, 2.0e5]
ALPHA_LIST = [0.05, 0.10, 0.20]

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0
    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic = 0
    n_pairs = 0
    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += (len(files) - 1)
    return n_ic, n_pairs

def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir = None
    best_pairs = -1
    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_pairs = n_pairs
            best_dir = d
    if best_dir is None or best_pairs <= 0:
        raise RuntimeError("No valid snapshot root found.")
    print(f"[AutoDetect] using: {best_dir}")
    return best_dir

_step_re = re.compile(r"klein_step_(\d+)\.npz")

def extract_step_from_path(path):
    m = _step_re.search(os.path.basename(path))
    if m is None:
        raise ValueError(f"Could not parse step from {path}")
    return int(m.group(1))

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def total_rmse_from_components(rmse_eta, rmse_u, rmse_v):
    return np.sqrt(rmse_eta**2 + rmse_u**2 + rmse_v**2)

def domain_mean_ke(eta, u, v, H=1000.0):
    h = H + eta
    ke = 0.5 * h * (u * u + v * v)
    return float(np.mean(ke))

def list_ic_dirs(data_dir):
    return sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)])

def list_ic_keys(data_dir):
    return [os.path.basename(d) for d in list_ic_dirs(data_dir)]

# ------------------------------------------------------------
# Discrete operators
# ------------------------------------------------------------
def laplacian_np(field, dx, dy):
    left  = np.roll(field,  1, axis=1)
    right = np.roll(field, -1, axis=1)
    d2x = (left - 2.0 * field + right) / (dx * dx)

    d2y = np.zeros_like(field)
    d2y[1:-1, :] = (field[2:, :] - 2.0 * field[1:-1, :] + field[:-2, :]) / (dy * dy)
    d2y[0, :]    = (field[1, :] - field[0, :]) / (dy * dy)
    d2y[-1, :]   = (field[-2, :] - field[-1, :]) / (dy * dy)

    return d2x + d2y

def ddx_np(field, dx):
    left  = np.roll(field,  1, axis=1)
    right = np.roll(field, -1, axis=1)
    return (right - left) / (2.0 * dx)

def ddy_np(field, dy):
    out = np.zeros_like(field)
    out[1:-1, :] = (field[2:, :] - field[:-2, :]) / (2.0 * dy)
    out[0, :]    = (field[1, :] - field[0, :]) / dy
    out[-1, :]   = (field[-1, :] - field[-2, :]) / dy
    return out

def safe_coriolis(f, fmin=2.0e-5):
    sign = np.sign(f)
    sign[sign == 0.0] = 1.0
    return sign * np.maximum(np.abs(f), fmin)

def geostrophic_velocity_from_eta(eta, f, dx, dy, g=9.81, fmin=2.0e-5):
    etax = ddx_np(eta, dx)
    etay = ddy_np(eta, dy)
    f_safe = safe_coriolis(f, fmin=fmin)
    ug = -(g / f_safe) * etay
    vg =  (g / f_safe) * etax
    return ug, vg

# ------------------------------------------------------------
# Exact model
# ------------------------------------------------------------
class ResBlock(nn.Module):
    def __init__(self, ch, dilation=1):
        super().__init__()
        pad = dilation
        self.c1 = nn.Conv2d(ch, ch, kernel_size=3, padding=pad, dilation=dilation)
        self.c2 = nn.Conv2d(ch, ch, kernel_size=3, padding=pad, dilation=dilation)
        self.act = nn.GELU()

    def forward(self, x):
        r = x
        x = self.act(self.c1(x))
        x = self.c2(x)
        return self.act(x + r)

class MultiStepContinuousResCNNModel(nn.Module):
    def __init__(self, in_ch=3, feat_ch=96, hidden=192, n_blocks=8):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, feat_ch, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, kernel_size=3, padding=1),
            nn.GELU(),
        )

        blocks = []
        dilations = [1, 1, 2, 2, 1, 1, 2, 2, 1, 1, 2, 2]
        for i in range(n_blocks):
            blocks.append(ResBlock(feat_ch, dilation=dilations[i % len(dilations)]))
        self.resnet = nn.Sequential(*blocks)

        self.grid_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, 3, kernel_size=3, padding=1),
        )

        self.query_mlp = nn.Sequential(
            nn.Linear(feat_ch + 3 + 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

    def encode(self, x):
        x = self.stem(x)
        x = self.resnet(x)
        return x

    def grid_tendency(self, feat):
        return self.grid_head(feat)

    def _sample_local(self, field_1b, x_norm_detached, y_norm_detached):
        grid = torch.stack([x_norm_detached, y_norm_detached], dim=-1).view(1, -1, 1, 2)
        vals = F.grid_sample(
            field_1b, grid,
            mode="bilinear",
            padding_mode="border",
            align_corners=True
        )
        vals = vals.squeeze(0).squeeze(-1).transpose(0, 1)
        return vals

    def query_state(self, x_1b, feat_1b, x_norm, y_norm, tau):
        x_norm_s = x_norm.detach()
        y_norm_s = y_norm.detach()

        feat_local = self._sample_local(feat_1b, x_norm_s, y_norm_s)
        state0_local = self._sample_local(x_1b, x_norm_s, y_norm_s)

        q = torch.stack([x_norm, y_norm, tau], dim=-1)
        inp = torch.cat([feat_local, state0_local, q], dim=-1)

        delta = self.query_mlp(inp)
        state_hat = state0_local + tau.unsqueeze(-1) * delta
        return state_hat

# ------------------------------------------------------------
# Load checkpoints
# ------------------------------------------------------------

loaded_runs = load_all_models(RUNS)

def infer_model_dims_from_ckpt(ckpt):
    cfg_ckpt = ckpt.get("config", {})

    # N_BLOCKS
    n_blocks = cfg_ckpt.get("N_BLOCKS", None)
    if n_blocks is None:
        idxs = []
        for k in ckpt["model_state_dict"].keys():
            m = re.match(r"resnet\.(\d+)\.", k)
            if m:
                idxs.append(int(m.group(1)))
        n_blocks = max(idxs) + 1 if len(idxs) > 0 else 8
    n_blocks = int(n_blocks)

    # FEAT_CH
    feat_ch = cfg_ckpt.get("FEAT_CH", None)
    if feat_ch is None:
        feat_ch = ckpt["model_state_dict"]["stem.0.weight"].shape[0]
    feat_ch = int(feat_ch)

    # HIDDEN
    hidden = cfg_ckpt.get("HIDDEN", None)
    if hidden is None:
        hidden = ckpt["model_state_dict"]["query_mlp.0.weight"].shape[0]
    hidden = int(hidden)

    return n_blocks, feat_ch, hidden


def load_all_models(run_defs):
    loaded = []

    for rd in run_defs:
        ckpt_path = os.path.join(rd["ckpt_dir"], rd["ckpt_name"])
        if not os.path.exists(ckpt_path):
            raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

        ckpt = torch.load(ckpt_path, map_location=device)

        n_blocks, feat_ch, hidden = infer_model_dims_from_ckpt(ckpt)

        model = MultiStepContinuousResCNNModel(
            feat_ch=feat_ch,
            hidden=hidden,
            n_blocks=n_blocks,
        ).to(device)

        model.load_state_dict(ckpt["model_state_dict"])
        model.eval()

        data_dir = ckpt.get("data_dir", None)

        loaded.append({
            "name": rd["name"],
            "ckpt_path": ckpt_path,
            "data_dir_from_ckpt": data_dir,
            "model": model,
            "n_blocks": n_blocks,
            "feat_ch": feat_ch,
            "hidden": hidden,
        })

        print(
            f"[Load] {rd['name']} <- {ckpt_path} | "
            f"N_BLOCKS={n_blocks}, FEAT_CH={feat_ch}, HIDDEN={hidden}"
        )

    return loaded

# ------------------------------------------------------------
# Locate FD data
# ------------------------------------------------------------
data_dir = None
for rr in loaded_runs:
    dd = rr.get("data_dir_from_ckpt", None)
    if (dd is not None) and os.path.exists(dd):
        data_dir = dd
        break

if data_dir is None:
    data_dir = autodetect_data_dir(DATA_DIR_CANDIDATES)

print("[Eval] using FD data dir:", data_dir)

available_keys = list_ic_keys(data_dir)
print(f"[Eval] found {len(available_keys)} ICs")
print("[Eval] available ICs:", available_keys)

# ------------------------------------------------------------
# Select cases
# ------------------------------------------------------------
if MODE == "one":
    selected_cases = [ONE_CASE]
elif MODE == "list":
    selected_cases = CASE_LIST[:]
elif MODE == "all":
    selected_cases = available_keys[:]
else:
    raise ValueError(f"Unknown MODE={MODE}")

selected_cases = [k for k in selected_cases if k in available_keys]
if len(selected_cases) == 0:
    raise RuntimeError("No valid selected cases found.")

print("[Eval] selected cases:", selected_cases)

# ------------------------------------------------------------
# Truth loader
# ------------------------------------------------------------
def load_truth_case(ic_key):
    ic_dir = os.path.join(data_dir, ic_key)
    files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))

    n_avail = len(files) - START_INDEX - 1
    if n_avail <= 0:
        return None

    rollout_steps = min(ROLLOUT_STEPS, n_avail)

    truth_seq = []
    truth_times = []
    truth_steps = []
    f_ref = None

    for fpath in files[START_INDEX:START_INDEX + rollout_steps + 1]:
        z = np.load(fpath)
        truth_seq.append(np.stack([z["eta"], z["uc"], z["vc"]], axis=0).astype(np.float32))
        truth_times.append(float(z["t"]))
        truth_steps.append(int(extract_step_from_path(fpath)))
        if f_ref is None:
            f_ref = z["f"].astype(np.float32)

    truth_seq = np.stack(truth_seq, axis=0)
    truth_times = np.array(truth_times, dtype=np.float32)
    truth_steps = np.array(truth_steps, dtype=np.int32)

    roll_steps = np.arange(1, rollout_steps + 1)
    roll_hours = roll_steps * DT_MACRO / 3600.0

    return {
        "ic_key": ic_key,
        "rollout_steps": rollout_steps,
        "truth_seq": truth_seq,
        "truth_times": truth_times,
        "truth_steps": truth_steps,
        "roll_hours": roll_hours,
        "f_ref": f_ref,
    }

# ------------------------------------------------------------
# Rollout
# ------------------------------------------------------------
def rollout_case(model, truth_pack, method_name="plain", nu=0.0, alpha=0.0):
    truth_seq = truth_pack["truth_seq"]
    rollout_steps = truth_pack["rollout_steps"]
    f_ref = truth_pack["f_ref"]
    roll_hours = truth_pack["roll_hours"]

    x = torch.tensor(truth_seq[0], dtype=torch.float32, device=device).unsqueeze(0)
    pred_seq = [truth_seq[0].copy()]

    with torch.no_grad():
        for k in range(rollout_steps):
            feat = model.encode(x)
            xdot = model.grid_tendency(feat)

            x_np = x[0].detach().cpu().numpy()
            xdot_np = xdot[0].detach().cpu().numpy()

            eta = x_np[0]
            u   = x_np[1]
            v   = x_np[2]

            eta_dot = xdot_np[0]
            u_dot   = xdot_np[1]
            v_dot   = xdot_np[2]

            eta_ml = eta + DT_MACRO * eta_dot
            u_ml   = u   + DT_MACRO * u_dot
            v_ml   = v   + DT_MACRO * v_dot

            if method_name == "diffusion":
                u_ml = u_ml + DT_MACRO * nu * laplacian_np(u, DX, DY)
                v_ml = v_ml + DT_MACRO * nu * laplacian_np(v, DX, DY)

            elif method_name == "geonudge":
                ug, vg = geostrophic_velocity_from_eta(
                    eta_ml, f_ref, DX, DY, g=G, fmin=FMIN
                )
                u_ml = (1.0 - alpha) * u_ml + alpha * ug
                v_ml = (1.0 - alpha) * v_ml + alpha * vg

            x_next_np = np.empty_like(x_np)
            x_next_np[0] = eta_ml
            x_next_np[1] = u_ml
            x_next_np[2] = v_ml

            x = torch.tensor(x_next_np, dtype=torch.float32, device=device).unsqueeze(0)
            pred_seq.append(x_next_np.copy())

    pred_seq = np.stack(pred_seq, axis=0)

    rmse_eta = []
    rmse_u   = []
    rmse_v   = []
    ke_fd    = []
    ke_ml    = []

    for k in range(1, rollout_steps + 1):
        eta_fd = truth_seq[k, 0]
        u_fd   = truth_seq[k, 1]
        v_fd   = truth_seq[k, 2]

        eta_ml = pred_seq[k, 0]
        u_ml   = pred_seq[k, 1]
        v_ml   = pred_seq[k, 2]

        rmse_eta.append(rmse(eta_ml, eta_fd))
        rmse_u.append(rmse(u_ml, u_fd))
        rmse_v.append(rmse(v_ml, v_fd))

        ke_fd.append(domain_mean_ke(eta_fd, u_fd, v_fd, H=H))
        ke_ml.append(domain_mean_ke(eta_ml, u_ml, v_ml, H=H))

    rmse_eta = np.array(rmse_eta)
    rmse_u   = np.array(rmse_u)
    rmse_v   = np.array(rmse_v)
    ke_fd    = np.array(ke_fd)
    ke_ml    = np.array(ke_ml)

    return {
        "method": method_name,
        "nu": nu,
        "alpha": alpha,
        "roll_hours": roll_hours,
        "truth_seq": truth_seq,
        "pred_seq": pred_seq,
        "rmse_eta": rmse_eta,
        "rmse_u": rmse_u,
        "rmse_v": rmse_v,
        "rmse_total": total_rmse_from_components(rmse_eta, rmse_u, rmse_v),
        "ke_fd": ke_fd,
        "ke_ml": ke_ml,
    }

def method_label(method_name, nu=0.0, alpha=0.0):
    if method_name == "plain":
        return "plain"
    if method_name == "diffusion":
        return f"diff_nu_{nu:.1e}"
    if method_name == "geonudge":
        return f"geonudge_a_{alpha:.2f}"
    return method_name

# ------------------------------------------------------------
# Run all evaluations
# ------------------------------------------------------------
summary_rows = []
all_results = {}  # all_results[ic_key][run_name][label] = result

for ic_key in selected_cases:
    print(f"\n[Eval] case: {ic_key}")

    truth_pack = load_truth_case(ic_key)
    if truth_pack is None:
        print(f"[Skip] {ic_key}: no usable truth sequence.")
        continue

    all_results[ic_key] = {}

    for rr in loaded_runs:
        run_name = rr["name"]
        model = rr["model"]

        print(f"  [Run] {run_name}")
        all_results[ic_key][run_name] = {}

        method_specs = []
        if RUN_PLAIN:
            method_specs.append(("plain", 0.0, 0.0))
        if RUN_DIFFUSION:
            for nu in NU_LIST:
                method_specs.append(("diffusion", nu, 0.0))
        if RUN_GEONUDGE:
            for alpha in ALPHA_LIST:
                method_specs.append(("geonudge", 0.0, alpha))

        for method_name, nu, alpha in method_specs:
            result = rollout_case(
                model=model,
                truth_pack=truth_pack,
                method_name=method_name,
                nu=nu,
                alpha=alpha
            )

            label = method_label(method_name, nu=nu, alpha=alpha)
            all_results[ic_key][run_name][label] = result

            final_ke_ratio = float(result["ke_ml"][-1] / max(result["ke_fd"][-1], 1e-12))

            summary_rows.append({
                "ic_key": ic_key,
                "run_name": run_name,
                "method": method_name,
                "nu": nu if method_name == "diffusion" else "",
                "alpha": alpha if method_name == "geonudge" else "",
                "rollout_steps": int(truth_pack["rollout_steps"]),
                "final_eta_rmse": float(result["rmse_eta"][-1]),
                "final_u_rmse": float(result["rmse_u"][-1]),
                "final_v_rmse": float(result["rmse_v"][-1]),
                "final_total_rmse": float(result["rmse_total"][-1]),
                "final_fd_ke": float(result["ke_fd"][-1]),
                "final_ml_ke": float(result["ke_ml"][-1]),
                "final_ke_ratio": final_ke_ratio,
            })

            print(
                f"    {label:20s} | "
                f"eta={result['rmse_eta'][-1]:.4f}  "
                f"u={result['rmse_u'][-1]:.4f}  "
                f"v={result['rmse_v'][-1]:.4f}  "
                f"tot={result['rmse_total'][-1]:.4f}  "
                f"KE ratio={final_ke_ratio:.4e}"
            )

# ------------------------------------------------------------
# Save summary CSV
# ------------------------------------------------------------
if len(summary_rows) > 0:
    summary_csv = os.path.join(EVAL_DIR, "summary_eval_compare_t6_keweak_runs.csv")
    keys = list(summary_rows[0].keys())
    with open(summary_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for row in summary_rows:
            w.writerow(row)
    print("\n[Eval] wrote summary CSV:", summary_csv)
else:
    print("\n[Eval] no successful evaluations.")

# ------------------------------------------------------------
# Save best-by-case-and-run table
# ------------------------------------------------------------
if len(summary_rows) > 0:
    best_rows = []
    for ic_key in all_results.keys():
        for run_name in all_results[ic_key].keys():
            candidates = [r for r in summary_rows if r["ic_key"] == ic_key and r["run_name"] == run_name]
            if len(candidates) == 0:
                continue
            best = min(candidates, key=lambda r: r["final_total_rmse"])
            best_rows.append(best)

    if len(best_rows) > 0:
        best_csv = os.path.join(EVAL_DIR, "best_method_per_case_and_run.csv")
        keys = list(best_rows[0].keys())
        with open(best_csv, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            for row in best_rows:
                w.writerow(row)
        print("[Eval] wrote best-method CSV:", best_csv)

# ------------------------------------------------------------
# Plot helpers
# ------------------------------------------------------------
def save_plot(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)

# ------------------------------------------------------------
# Per-case plots comparing RUNS for SAME method
# ------------------------------------------------------------
for ic_key, run_pack in all_results.items():
    labels = []
    for run_name in run_pack.keys():
        for lab in run_pack[run_name].keys():
            if lab not in labels:
                labels.append(lab)

    for lab in labels:
        present_run_names = [rn for rn in run_pack.keys() if lab in run_pack[rn]]
        if len(present_run_names) == 0:
            continue

        sample = run_pack[present_run_names[0]][lab]
        hrs = sample["roll_hours"]

        # total RMSE
        fig = plt.figure(figsize=(9, 5))
        for run_name in present_run_names:
            res = run_pack[run_name][lab]
            plt.plot(hrs, res["rmse_total"], marker="o", label=run_name)
        plt.xlabel("Rollout time (hours)")
        plt.ylabel("Combined RMSE")
        plt.title(f"Combined RMSE vs time | {ic_key} | {lab}")
        plt.legend()
        save_plot(fig, os.path.join(EVAL_DIR, f"{ic_key}_{lab}_compare_runs_total_rmse.png"))

        # eta RMSE
        fig = plt.figure(figsize=(9, 5))
        for run_name in present_run_names:
            res = run_pack[run_name][lab]
            plt.plot(hrs, res["rmse_eta"], marker="o", label=run_name)
        plt.xlabel("Rollout time (hours)")
        plt.ylabel("eta RMSE")
        plt.title(f"eta RMSE vs time | {ic_key} | {lab}")
        plt.legend()
        save_plot(fig, os.path.join(EVAL_DIR, f"{ic_key}_{lab}_compare_runs_eta_rmse.png"))

        # u RMSE
        fig = plt.figure(figsize=(9, 5))
        for run_name in present_run_names:
            res = run_pack[run_name][lab]
            plt.plot(hrs, res["rmse_u"], marker="o", label=run_name)
        plt.xlabel("Rollout time (hours)")
        plt.ylabel("u RMSE")
        plt.title(f"u RMSE vs time | {ic_key} | {lab}")
        plt.legend()
        save_plot(fig, os.path.join(EVAL_DIR, f"{ic_key}_{lab}_compare_runs_u_rmse.png"))

        # v RMSE
        fig = plt.figure(figsize=(9, 5))
        for run_name in present_run_names:
            res = run_pack[run_name][lab]
            plt.plot(hrs, res["rmse_v"], marker="o", label=run_name)
        plt.xlabel("Rollout time (hours)")
        plt.ylabel("v RMSE")
        plt.title(f"v RMSE vs time | {ic_key} | {lab}")
        plt.legend()
        save_plot(fig, os.path.join(EVAL_DIR, f"{ic_key}_{lab}_compare_runs_v_rmse.png"))

        # KE
        fig = plt.figure(figsize=(9, 5))
        plt.plot(hrs, sample["ke_fd"], linewidth=2.5, label="FD KE")
        for run_name in present_run_names:
            res = run_pack[run_name][lab]
            plt.plot(hrs, res["ke_ml"], marker="o", label=run_name)
        plt.xlabel("Rollout time (hours)")
        plt.ylabel("Domain-mean kinetic energy")
        plt.title(f"KE vs time | {ic_key} | {lab}")
        plt.legend()
        save_plot(fig, os.path.join(EVAL_DIR, f"{ic_key}_{lab}_compare_runs_ke.png"))

# ------------------------------------------------------------
# Per-case plots comparing METHODS for SAME run
# ------------------------------------------------------------
for ic_key, run_pack in all_results.items():
    for run_name, method_pack in run_pack.items():
        labels = list(method_pack.keys())
        if len(labels) == 0:
            continue

        sample = method_pack[labels[0]]
        hrs = sample["roll_hours"]

        # total RMSE
        fig = plt.figure(figsize=(10, 5))
        for lab in labels:
            res = method_pack[lab]
            plt.plot(hrs, res["rmse_total"], marker="o", label=lab)
        plt.xlabel("Rollout time (hours)")
        plt.ylabel("Combined RMSE")
        plt.title(f"Combined RMSE vs time | {ic_key} | {run_name}")
        plt.legend(fontsize=8)
        save_plot(fig, os.path.join(EVAL_DIR, f"{ic_key}_{run_name}_compare_methods_total_rmse.png"))

        # KE
        fig = plt.figure(figsize=(10, 5))
        plt.plot(hrs, sample["ke_fd"], linewidth=2.5, label="FD KE")
        for lab in labels:
            res = method_pack[lab]
            plt.plot(hrs, res["ke_ml"], marker="o", label=lab)
        plt.xlabel("Rollout time (hours)")
        plt.ylabel("Domain-mean kinetic energy")
        plt.title(f"KE vs time | {ic_key} | {run_name}")
        plt.legend(fontsize=8)
        save_plot(fig, os.path.join(EVAL_DIR, f"{ic_key}_{run_name}_compare_methods_ke.png"))

# ------------------------------------------------------------
# Aggregate CSV + plots
# ------------------------------------------------------------
if len(summary_rows) > 0:
    method_keys = []
    for row in summary_rows:
        lab = method_label(
            row["method"],
            row["nu"] if row["nu"] != "" else 0.0,
            row["alpha"] if row["alpha"] != "" else 0.0
        )
        if lab not in method_keys:
            method_keys.append(lab)

    agg = {}
    for row in summary_rows:
        run_name = row["run_name"]
        lab = method_label(
            row["method"],
            row["nu"] if row["nu"] != "" else 0.0,
            row["alpha"] if row["alpha"] != "" else 0.0
        )
        agg.setdefault(run_name, {})
        agg[run_name].setdefault(lab, {
            "final_total_rmse": [],
            "final_ke_ratio": [],
            "final_eta_rmse": [],
            "final_u_rmse": [],
            "final_v_rmse": [],
        })
        agg[run_name][lab]["final_total_rmse"].append(row["final_total_rmse"])
        agg[run_name][lab]["final_ke_ratio"].append(row["final_ke_ratio"])
        agg[run_name][lab]["final_eta_rmse"].append(row["final_eta_rmse"])
        agg[run_name][lab]["final_u_rmse"].append(row["final_u_rmse"])
        agg[run_name][lab]["final_v_rmse"].append(row["final_v_rmse"])

    agg_rows = []
    for run_name in agg:
        for lab in agg[run_name]:
            dd = agg[run_name][lab]
            agg_rows.append({
                "run_name": run_name,
                "label": lab,
                "mean_final_total_rmse": float(np.mean(dd["final_total_rmse"])),
                "mean_final_ke_ratio": float(np.mean(dd["final_ke_ratio"])),
                "mean_final_eta_rmse": float(np.mean(dd["final_eta_rmse"])),
                "mean_final_u_rmse": float(np.mean(dd["final_u_rmse"])),
                "mean_final_v_rmse": float(np.mean(dd["final_v_rmse"])),
            })

    agg_csv = os.path.join(EVAL_DIR, "aggregate_mean_metrics_by_run_and_method.csv")
    with open(agg_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(agg_rows[0].keys()))
        w.writeheader()
        for row in agg_rows:
            w.writerow(row)
    print("[Eval] wrote aggregate CSV:", agg_csv)

    # total RMSE bar
    fig = plt.figure(figsize=(12, 6))
    x_labels = []
    y_vals = []
    for row in agg_rows:
        x_labels.append(f"{row['run_name']}\n{row['label']}")
        y_vals.append(row["mean_final_total_rmse"])
    plt.bar(np.arange(len(y_vals)), y_vals)
    plt.xticks(np.arange(len(y_vals)), x_labels, rotation=90)
    plt.ylabel("Mean final total RMSE")
    plt.title("Mean final total RMSE across selected cases")
    save_plot(fig, os.path.join(EVAL_DIR, "aggregate_mean_final_total_rmse.png"))

    # KE ratio bar
    fig = plt.figure(figsize=(12, 6))
    x_labels = []
    y_vals = []
    for row in agg_rows:
        x_labels.append(f"{row['run_name']}\n{row['label']}")
        y_vals.append(row["mean_final_ke_ratio"])
    plt.bar(np.arange(len(y_vals)), y_vals)
    plt.xticks(np.arange(len(y_vals)), x_labels, rotation=90)
    plt.ylabel("Mean final KE ratio")
    plt.title("Mean final KE ratio across selected cases")
    save_plot(fig, os.path.join(EVAL_DIR, "aggregate_mean_final_ke_ratio.png"))

# ------------------------------------------------------------
# Optional field plots
# ------------------------------------------------------------
if MAKE_FIELD_PLOTS:
    for ic_key, run_pack in all_results.items():
        if FIELD_PLOT_RUN_NAME not in run_pack:
            continue
        if FIELD_PLOT_METHOD not in run_pack[FIELD_PLOT_RUN_NAME]:
            continue

        result = run_pack[FIELD_PLOT_RUN_NAME][FIELD_PLOT_METHOD]
        truth_seq = result["truth_seq"]
        pred_seq = result["pred_seq"]
        rollout_steps = len(result["rmse_eta"])

        for kk in FIELD_PLOT_STEPS:
            if kk < 0 or kk >= rollout_steps:
                continue

            t_idx = kk + 1

            eta_fd = truth_seq[t_idx, 0]
            u_fd   = truth_seq[t_idx, 1]
            v_fd   = truth_seq[t_idx, 2]

            eta_ml = pred_seq[t_idx, 0]
            u_ml   = pred_seq[t_idx, 1]
            v_ml   = pred_seq[t_idx, 2]

            fig, axes = plt.subplots(3, 3, figsize=(14, 11))

            im = axes[0, 0].imshow(eta_fd, origin="lower")
            axes[0, 0].set_title(f"FD eta | step {t_idx}")
            plt.colorbar(im, ax=axes[0, 0], fraction=0.046, pad=0.04)

            im = axes[0, 1].imshow(eta_ml, origin="lower")
            axes[0, 1].set_title(f"{FIELD_PLOT_RUN_NAME} {FIELD_PLOT_METHOD} eta")
            plt.colorbar(im, ax=axes[0, 1], fraction=0.046, pad=0.04)

            im = axes[0, 2].imshow(eta_ml - eta_fd, origin="lower")
            axes[0, 2].set_title("ML - FD eta")
            plt.colorbar(im, ax=axes[0, 2], fraction=0.046, pad=0.04)

            im = axes[1, 0].imshow(u_fd, origin="lower")
            axes[1, 0].set_title("FD u")
            plt.colorbar(im, ax=axes[1, 0], fraction=0.046, pad=0.04)

            im = axes[1, 1].imshow(u_ml, origin="lower")
            axes[1, 1].set_title("ML u")
            plt.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)

            im = axes[1, 2].imshow(u_ml - u_fd, origin="lower")
            axes[1, 2].set_title("ML - FD u")
            plt.colorbar(im, ax=axes[1, 2], fraction=0.046, pad=0.04)

            im = axes[2, 0].imshow(v_fd, origin="lower")
            axes[2, 0].set_title("FD v")
            plt.colorbar(im, ax=axes[2, 0], fraction=0.046, pad=0.04)

            im = axes[2, 1].imshow(v_ml, origin="lower")
            axes[2, 1].set_title("ML v")
            plt.colorbar(im, ax=axes[2, 1], fraction=0.046, pad=0.04)

            im = axes[2, 2].imshow(v_ml - v_fd, origin="lower")
            axes[2, 2].set_title("ML - FD v")
            plt.colorbar(im, ax=axes[2, 2], fraction=0.046, pad=0.04)

            plt.tight_layout()
            out_name = f"{ic_key}_{FIELD_PLOT_RUN_NAME}_{FIELD_PLOT_METHOD}_fields_step_{t_idx:03d}.png"
            plt.savefig(os.path.join(EVAL_DIR, out_name), dpi=150)
            plt.close()

print("\n[Eval] finished.")
print("[Eval] outputs saved in:", EVAL_DIR)

Using device: cuda
GPU: Tesla T4
[Load] b12_c96_t6_p4_keweak <- /content/drive/MyDrive/AI_4DVAR2/training_runs/rescnn_b12_c96_lr1e4_t6_p4_keweak/final_model.pt | N_BLOCKS=12, FEAT_CH=96, HIDDEN=192
[Load] b12_c96_t6_p4_keweak_lam001 <- /content/drive/MyDrive/AI_4DVAR2/training_runs/rescnn_b12_c96_lr1e4_t6_p4_keweak_lam001/final_model.pt | N_BLOCKS=12, FEAT_CH=96, HIDDEN=192
[Load] b12_c128_t6_p4_keweak <- /content/drive/MyDrive/AI_4DVAR2/training_runs/rescnn_b12_c128_lr1e4_t6_p4_keweak/final_model.pt | N_BLOCKS=12, FEAT_CH=128, HIDDEN=256
[Eval] using FD data dir: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
[Eval] found 28 ICs
[Eval] available ICs: ['gauss_00', 'gauss_01', 'gauss_02', 'gauss_03', 'gauss_04', 'gauss_05', 'mix_00', 'mix_01', 'mix_02', 'mix_03', 'mix_04', 'mix_05', 'test_modon_00', 'test_modon_01', 'test_rh_00', 'test_rh_01', 'vort_00', 'vort_01', 'vort_02', 'vort_03', 'vort_04', 'vort_05', 'wave_00', 'wave_01', 'wave_02', 'wave_03', 'wave_04', 'wave_05']
[Eval] selecte